In [ ]:
!pip install -q huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import os

token = userdata.get("HF_TOKEN")

zip_path = hf_hub_download(
    repo_id="Logic-Loop/drivejepa-nuscenes",
    filename="data.zip",
    repo_type="dataset",
    token=token
)

print("Dataset downloaded successfully!")
print("Location:", zip_path)
print("Size (GB):", round(os.path.getsize(zip_path) / (1024**3), 2))

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import os

# Dataset repository details
dataset_repository = "Logic-Loop/drivejepa-nuscenes"
dataset_zip_filename = "data.zip"

# Retrieve the access token securely
huggingface_token = userdata.get("HF_TOKEN")

# Download the archive into the Colab cache
dataset_zip_path = hf_hub_download(
    repo_id=dataset_repository,
    filename=dataset_zip_filename,
    repo_type="dataset",
    token=huggingface_token
)

print("Dataset is available!")
print("Dataset location:", dataset_zip_path)
print(
    "Dataset size:",
    round(os.path.getsize(dataset_zip_path) / (1024 ** 3), 2),
    "GiB"
)

In [ ]:
import zipfile
from collections import Counter

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    archive_file_names = [
        file_name
        for file_name in dataset_archive.namelist()
        if not file_name.endswith("/")
    ]

    print("Total files in archive:", len(archive_file_names))

    print("\nFirst 40 paths:")
    for file_name in archive_file_names[:40]:
        print(file_name)

    top_level_folders = Counter(
        file_name.split("/")[0]
        for file_name in archive_file_names
        if "/" in file_name
    )

    print("\nTop-level folders:")
    for folder_name, file_count in top_level_folders.most_common(20):
        print(f"{folder_name}: {file_count} files")

In [ ]:
# Read manifest and audit dataset conditions

import json

manifest_path = "data/manifest.json"

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:
    if manifest_path not in archive_file_names:
        raise FileNotFoundError(
            f"Manifest not found: {manifest_path}"
        )

    dataset_manifest = json.loads(
        dataset_archive.read(manifest_path).decode("utf-8")
    )

In [ ]:
# ============================================================
# STEP 6: Read manifest and audit dataset conditions
# Continue from the existing notebook
# ============================================================

manifest_path = "data/manifest.json"

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:
    if manifest_path not in archive_file_names:
        raise FileNotFoundError(
            f"Manifest not found: {manifest_path}"
        )

    dataset_manifest = json.loads(
        dataset_archive.read(manifest_path).decode("utf-8")
    )

episode_names = dataset_manifest.get("episodes", [])
episode_metadata = dataset_manifest.get("meta", {})

print("Total episodes:", len(episode_names))
print("Episodes with metadata:", len(episode_metadata))

# Count every recorded weather and lighting condition.
weather_counts = Counter()
lighting_counts = Counter()
rain_counts = Counter()
combined_condition_counts = Counter()

for episode_name in episode_names:
    episode_info = episode_metadata.get(episode_name, {})

    weather = episode_info.get("weather", "unknown")
    night = episode_info.get("night", "unknown")
    rain = episode_info.get("rain", "unknown")

    weather_counts[weather] += 1
    lighting_counts[str(night)] += 1
    rain_counts[str(rain)] += 1

    combined_condition_counts[
        (weather, str(night), str(rain))
    ] += 1

print("\n" + "=" * 65)
print("WEATHER CONDITIONS")
print("=" * 65)

for weather, count in sorted(weather_counts.items()):
    print(f"{weather:22} {count:4} episodes")

print("\n" + "=" * 65)
print("DAY/NIGHT METADATA")
print("=" * 65)

for lighting, count in sorted(lighting_counts.items()):
    print(f"night={lighting:8} {count:4} episodes")

print("\n" + "=" * 65)
print("RAIN METADATA")
print("=" * 65)

for rain, count in sorted(rain_counts.items()):
    print(f"rain={rain:8} {count:4} episodes")

print("\n" + "=" * 65)
print("COMBINED WEATHER AND LIGHTING CONDITIONS")
print("=" * 65)

for (weather, night, rain), count in sorted(
    combined_condition_counts.items()
):
    print(
        f"Weather: {weather:20} | "
        f"Night: {night:5} | "
        f"Rain: {rain:5} | "
        f"Episodes: {count}"
    )

In [ ]:
# Libraries used by the later cells
import math
import random
from io import BytesIO
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# ============================================================
# STEP 8: Create a reproducible 20% weather-balanced smoke test
# ============================================================

smoke_test_fraction = 0.20
random_seed = 42

random_generator = random.Random(random_seed)

# Group episodes by their weather label.
episodes_by_weather = {}

for episode_name in episode_names:
    episode_info = episode_metadata[episode_name]
    weather_name = episode_info.get("weather", "unknown")

    episodes_by_weather.setdefault(weather_name, []).append(
        episode_name
    )

# Shuffle each group reproducibly.
for weather_name in episodes_by_weather:
    random_generator.shuffle(episodes_by_weather[weather_name])

# Select 20% of the complete dataset.
target_episode_count = math.ceil(
    len(episode_names) * smoke_test_fraction
)

# Allocate the target sample proportionally across weather categories.
weather_allocations = {
    weather_name: len(weather_episodes) * smoke_test_fraction
    for weather_name, weather_episodes in episodes_by_weather.items()
}

weather_quotas = {
    weather_name: math.floor(allocation)
    for weather_name, allocation in weather_allocations.items()
}

remaining_slots = target_episode_count - sum(weather_quotas.values())

# Assign remaining slots to categories with the largest fractional remainders.
weather_priority = sorted(
    weather_allocations,
    key=lambda weather_name: (
        weather_allocations[weather_name]
        - weather_quotas[weather_name]
    ),
    reverse=True
)

for weather_name in weather_priority:
    if remaining_slots == 0:
        break

    if weather_quotas[weather_name] < len(
        episodes_by_weather[weather_name]
    ):
        weather_quotas[weather_name] += 1
        remaining_slots -= 1

# Sample episodes according to their weather allocations.
smoke_test_episode_names = []

for weather_name, weather_episodes in episodes_by_weather.items():
    quota = weather_quotas[weather_name]

    smoke_test_episode_names.extend(
        weather_episodes[:quota]
    )

# Keep output ordering stable.
smoke_test_episode_names.sort()

print("=" * 70)
print("DRIVEJEPA — 20% SMOKE TEST")
print("=" * 70)
print("Total episodes:", len(episode_names))
print("Selected episodes:", len(smoke_test_episode_names))
print("Target sample:", f"{smoke_test_fraction:.0%}")
print(
    "Actual sample:",
    f"{100 * len(smoke_test_episode_names) / len(episode_names):.2f}%"
)

print("\nWEATHER COVERAGE")
print("-" * 70)

selected_weather_counts = Counter(
    episode_metadata[episode_name].get("weather", "unknown")
    for episode_name in smoke_test_episode_names
)

for weather_name in sorted(episodes_by_weather):
    print(
        f"{weather_name:22} "
        f"{selected_weather_counts[weather_name]:2} / "
        f"{len(episodes_by_weather[weather_name]):2} episodes"
    )

print("\nSelected episode names:")
print(smoke_test_episode_names)

In [ ]:
# ============================================================
# STEP 9: Visualize all available weather categories
# ============================================================

representative_episodes = {}

for episode_name in smoke_test_episode_names:
    episode_info = episode_metadata[episode_name]
    weather_name = episode_info.get("weather", "unknown")

    # Keep one representative episode for each weather category.
    representative_episodes.setdefault(
        weather_name, episode_name
    )

weather_examples = sorted(representative_episodes.items())

number_of_columns = 3
number_of_rows = math.ceil(
    len(weather_examples) / number_of_columns
)

figure, image_axes = plt.subplots(
    number_of_rows,
    number_of_columns,
    figsize=(18, number_of_rows * 4),
    squeeze=False
)

image_axes = list(image_axes.flat)

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:
    for image_axis, (weather_name, episode_name) in zip(
        image_axes, weather_examples
    ):
        episode_info = episode_metadata[episode_name]

        frame_count = episode_info.get("n_frames", 1)
        middle_frame_index = max(0, frame_count // 2)

        frame_path = (
            f"data/{episode_name}/frames/"
            f"{middle_frame_index:06d}.jpg"
        )

        try:
            with Image.open(
                BytesIO(dataset_archive.read(frame_path))
            ) as source_image:
                image_axis.imshow(source_image.convert("RGB"))

            image_axis.set_title(
                f"{weather_name}\n"
                f"Episode: {episode_name}\n"
                f"Night: {episode_info.get('night', 'unknown')} | "
                f"Rain: {episode_info.get('rain', 'unknown')}",
                fontsize=10
            )

        except KeyError:
            image_axis.set_title(
                f"{weather_name}\nImage not found: {episode_name}"
            )

        image_axis.axis("off")

for image_axis in image_axes[len(weather_examples):]:
    image_axis.axis("off")

plt.suptitle(
    "DriveJEPA Smoke Test — Weather Coverage",
    fontsize=18
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 11: Inspect frames and metadata for the 20% smoke test
# ============================================================

from collections import defaultdict

smoke_test_fraction = 0.20
weather_groups = defaultdict(list)

# Group episodes by weather, preserving their metadata.
for episode_name in dataset_manifest["episodes"]:
    episode_metadata = dataset_manifest["meta"].get(episode_name, {})
    weather_name = episode_metadata.get("weather", "unknown")
    weather_groups[weather_name].append(episode_name)

print("=" * 65)
print("DRIVEJEPA CAMERA AND WEATHER INSPECTION")
print("=" * 65)

print("Total episodes:", len(dataset_manifest["episodes"]))
print("Smoke-test target:", f"{smoke_test_fraction:.0%}")
print("Weather categories:", len(weather_groups))

for weather_name, episode_names in sorted(weather_groups.items()):
    print(f"{weather_name:20s}: {len(episode_names):3d} episodes")

# Inspect actual frame paths from one episode in each weather category.
with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    available_paths = set(dataset_archive.namelist())

    print("\nFRAME PATH INSPECTION")
    print("-" * 65)

    for weather_name, episode_names in sorted(weather_groups.items()):

        example_episode = episode_names[0]
        frame_prefix = f"data/{example_episode}/frames/"

        example_frames = sorted(
            path for path in available_paths
            if path.startswith(frame_prefix)
            and path.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        print(f"\nWeather: {weather_name}")
        print("Example episode:", example_episode)
        print("Frame count:", len(example_frames))

        if example_frames:
            print("First frame:", example_frames[0])
            print("Last frame:", example_frames[-1])

    # Look for additional image streams anywhere in the archive.
    print("\nPOSSIBLE SEPARATE CAMERA STREAMS")
    print("-" * 65)

    image_paths = [
        path for path in available_paths
        if path.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for keyword in ("front", "back", "rear", "camera", "sensor"):
        matching_paths = [
            path for path in image_paths
            if keyword in path.lower()
        ]

        print(f"{keyword:10s}: {len(matching_paths)} matching image paths")

        for path in matching_paths[:3]:
            print("   ", path)

In [ ]:
# ============================================================
# STEP 12: Create a weather-balanced smoke-test subset
# ============================================================

import math

SMOKE_TEST_FRACTION = 0.20

smoke_test_episodes = []
weather_summary = {}

# Select approximately 20% from every weather category.
# Evenly spaced selection avoids taking only the first few episodes.
for weather_name, episode_names in sorted(weather_groups.items()):

    episode_names = sorted(episode_names)
    n_total = len(episode_names)

    n_select = max(1, math.ceil(n_total * SMOKE_TEST_FRACTION))

    if n_select >= n_total:
        selected = episode_names
    else:
        indices = [
            round(i * (n_total - 1) / (n_select - 1))
            if n_select > 1 else (n_total - 1) // 2
            for i in range(n_select)
        ]

        selected = [episode_names[i] for i in indices]

    smoke_test_episodes.extend(selected)

    weather_summary[weather_name] = {
        "available": n_total,
        "selected": len(selected),
        "episodes": selected
    }

print("=" * 70)
print("DRIVEJEPA WEATHER-BALANCED SMOKE TEST")
print("=" * 70)

print("Total dataset episodes:", len(dataset_manifest["episodes"]))
print("Selected smoke-test episodes:", len(smoke_test_episodes))
print(
    "Overall selection:",
    f"{len(smoke_test_episodes) / len(dataset_manifest['episodes']) * 100:.2f}%"
)

print("\nSELECTION BY WEATHER")
print("-" * 70)

for weather_name, summary in weather_summary.items():
    print(
        f"{weather_name:20s} "
        f"{summary['selected']:3d}/{summary['available']:3d} episodes"
    )

print("\nTotal weather categories covered:", len(weather_summary))

# ============================================================
# Verify that selected episodes have frames in the archive.
# ============================================================

smoke_test_frame_examples = []

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    archive_names = set(dataset_archive.namelist())

    for weather_name, summary in weather_summary.items():

        # Use the middle selected episode as the representative example.
        episode_name = summary["episodes"][len(summary["episodes"]) // 2]

        prefix = f"data/{episode_name}/frames/"

        frame_paths = sorted(
            path for path in archive_names
            if path.startswith(prefix)
            and path.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        if not frame_paths:
            print(
                f"WARNING: No image frames found for {episode_name} "
                f"({weather_name})"
            )
            continue

        # Select a middle frame to represent this weather condition.
        frame_path = frame_paths[len(frame_paths) // 2]

        smoke_test_frame_examples.append({
            "weather": weather_name,
            "episode": episode_name,
            "frame_path": frame_path,
            "frame_count": len(frame_paths)
        })

print("\nFRAME VERIFICATION")
print("-" * 70)

for item in smoke_test_frame_examples:
    print(
        f"{item['weather']:20s} | "
        f"{item['episode']} | "
        f"{item['frame_count']} frames | "
        f"{item['frame_path']}"
    )

print("\nEpisodes selected:", len(smoke_test_episodes))
print("Weather examples verified:", len(smoke_test_frame_examples))

In [ ]:
# ============================================================
# STEP 13: Visualize one RGB frame from every weather category
# ============================================================

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    for ax, item in zip(axes, smoke_test_frame_examples):

        with dataset_archive.open(item["frame_path"]) as frame_file:
            image = Image.open(frame_file).convert("RGB")

            ax.imshow(image)

        ax.set_title(
            f"Weather: {item['weather']}\n"
            f"Episode: {item['episode']}\n"
            f"Frames: {item['frame_count']}",
            fontsize=9
        )

        ax.axis("off")

# Hide unused panels if any categories were missing.
for ax in axes[len(smoke_test_frame_examples):]:
    ax.axis("off")

plt.suptitle(
    "DriveJEPA Smoke Test — Weather Coverage",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 14: Validate smoke-test image and driving-data integrity
# ============================================================
import numpy as np
from io import BytesIO

required_arrays = [
    "timestamps",
    "states",
    "actions",
    "commands",
    "waypoints",
    "poses",
]

episode_validation_results = []

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    available_archive_paths = set(dataset_archive.namelist())

    for episode_name in smoke_test_episodes:

        episode_prefix = f"data/{episode_name}/"

        # Find this episode's RGB frames.
        frame_paths = sorted(
            path
            for path in available_archive_paths
            if path.startswith(episode_prefix + "frames/")
            and path.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        episode_log_path = episode_prefix + "log.npz"

        result = {
            "episode": episode_name,
            "weather": dataset_manifest.get("meta", {})
                                      .get(episode_name, {})
                                      .get("weather", "unknown"),
            "frame_count": len(frame_paths),
            "log_found": episode_log_path in available_archive_paths,
            "arrays_found": [],
            "missing_arrays": [],
            "array_lengths": {},
            "status": "CHECKING",
        }

        if not frame_paths:
            result["status"] = "NO_FRAMES"
            episode_validation_results.append(result)
            continue

        if not result["log_found"]:
            result["status"] = "NO_LOG"
            episode_validation_results.append(result)
            continue

        try:
            with dataset_archive.open(episode_log_path) as log_file:
                with np.load(BytesIO(log_file.read()), allow_pickle=False) as episode_log:

                    for array_name in required_arrays:

                        if array_name not in episode_log.files:
                            result["missing_arrays"].append(array_name)
                            continue

                        array_values = episode_log[array_name]

                        result["arrays_found"].append(array_name)

                        # Scalar split metadata is not a per-frame array.
                        if array_values.ndim > 0:
                            result["array_lengths"][array_name] = len(array_values)

                    frame_count = result["frame_count"]

                    # Compare each per-frame array against the image count.
                    length_mismatches = {
                        name: length
                        for name, length in result["array_lengths"].items()
                        if length != frame_count
                    }

                    if result["missing_arrays"]:
                        result["status"] = "MISSING_ARRAYS"
                    elif length_mismatches:
                        result["status"] = "LENGTH_MISMATCH"
                        result["length_mismatches"] = length_mismatches
                    else:
                        result["status"] = "PASS"

        except Exception as error:
            result["status"] = "READ_ERROR"
            result["error"] = str(error)

        episode_validation_results.append(result)


# ============================================================
# Summarize validation results
# ============================================================

validation_status_counts = Counter(
    result["status"] for result in episode_validation_results
)

print("=" * 75)
print("DRIVEJEPA SMOKE-TEST DATA VALIDATION")
print("=" * 75)

print("Episodes checked:", len(episode_validation_results))

for status_name, status_count in sorted(validation_status_counts.items()):
    print(f"{status_name:20s}: {status_count}")

print("\nEPISODE-LEVEL RESULTS")
print("-" * 75)

for result in episode_validation_results:

    print(
        f"{result['episode']:8s} | "
        f"{result['weather']:18s} | "
        f"frames={result['frame_count']:4d} | "
        f"log={str(result['log_found']):5s} | "
        f"{result['status']}"
    )

    if result.get("missing_arrays"):
        print("    Missing:", result["missing_arrays"])

    if result.get("length_mismatches"):
        print("    Length mismatches:", result["length_mismatches"])

    if result.get("error"):
        print("    Error:", result["error"])

print("\nVALIDATION COMPLETE")

In [ ]:
# ============================================================
# STEP 16: Inspect numerical driving data
# ============================================================

arrays_to_inspect = [
    "timestamps",
    "states",
    "actions",
    "commands",
    "waypoints",
    "poses",
]

numerical_data_summary = {}
invalid_value_report = []

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    for episode_name in smoke_test_episodes:

        log_path = f"data/{episode_name}/log.npz"

        with dataset_archive.open(log_path) as log_file:
            with np.load(BytesIO(log_file.read()), allow_pickle=False) as episode_log:

                for array_name in arrays_to_inspect:

                    if array_name not in episode_log.files:
                        continue

                    values = episode_log[array_name]

                    # Skip scalar metadata such as the split field.
                    if values.ndim == 0:
                        continue

                    values = np.asarray(values)

                    if not np.issubdtype(values.dtype, np.number):
                        continue

                    finite_mask = np.isfinite(values)
                    invalid_count = int(values.size - finite_mask.sum())

                    if array_name not in numerical_data_summary:
                        numerical_data_summary[array_name] = {
                            "shapes": Counter(),
                            "total_values": 0,
                            "invalid_values": 0,
                            "minimum": float("inf"),
                            "maximum": float("-inf"),
                            "episodes_checked": 0,
                        }

                    summary = numerical_data_summary[array_name]

                    summary["shapes"][str(values.shape)] += 1
                    summary["total_values"] += int(values.size)
                    summary["invalid_values"] += invalid_count
                    summary["episodes_checked"] += 1

                    if invalid_count:
                        invalid_value_report.append({
                            "episode": episode_name,
                            "array": array_name,
                            "invalid_values": invalid_count,
                        })

                    finite_values = values[finite_mask]

                    if finite_values.size:
                        summary["minimum"] = min(
                            summary["minimum"],
                            float(finite_values.min())
                        )
                        summary["maximum"] = max(
                            summary["maximum"],
                            float(finite_values.max())
                        )

# ============================================================
# Print summary
# ============================================================

print("=" * 75)
print("DRIVEJEPA NUMERICAL DATA INSPECTION")
print("=" * 75)

for array_name, summary in numerical_data_summary.items():

    minimum = summary["minimum"]
    maximum = summary["maximum"]

    if minimum == float("inf"):
        minimum = float("nan")

    if maximum == float("-inf"):
        maximum = float("nan")

    print(f"\nArray: {array_name}")
    print("-" * 50)
    print("Episodes inspected:", summary["episodes_checked"])
    print("Shapes:", dict(summary["shapes"]))
    print("Total values:", summary["total_values"])
    print("Non-finite values:", summary["invalid_values"])
    print("Minimum:", minimum)
    print("Maximum:", maximum)

print("\n" + "=" * 75)
print("INVALID-VALUE REPORT")
print("=" * 75)

if invalid_value_report:
    for issue in invalid_value_report[:30]:
        print(issue)

    if len(invalid_value_report) > 30:
        print("Additional issues:", len(invalid_value_report) - 30)
else:
    print("No NaN or infinite values found in the inspected numeric arrays.")

In [ ]:
# ============================================================
# STEP 17A: Analyze individual driving-data columns
# ============================================================

arrays_for_column_analysis = [
    "states",
    "actions",
    "waypoints",
    "poses",
]

column_statistics = {}

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    for episode_name in smoke_test_episodes:

        log_path = f"data/{episode_name}/log.npz"

        with dataset_archive.open(log_path) as log_file:
            with np.load(BytesIO(log_file.read()), allow_pickle=False) as episode_log:

                for array_name in arrays_for_column_analysis:

                    values = episode_log[array_name]

                    if values.ndim != 2:
                        continue

                    if array_name not in column_statistics:
                        column_statistics[array_name] = {
                            "columns": [[] for _ in range(values.shape[1])]
                        }

                    for column_index in range(values.shape[1]):
                        column_statistics[array_name]["columns"][column_index].append(
                            values[:, column_index]
                        )

print("=" * 75)
print("INDIVIDUAL COLUMN STATISTICS")
print("=" * 75)

for array_name, array_summary in column_statistics.items():

    print(f"\n{array_name.upper()}")
    print("-" * 75)

    for column_index, column_parts in enumerate(
        array_summary["columns"]
    ):

        column_values = np.concatenate(column_parts)

        print(
            f"Column {column_index}: "
            f"min={column_values.min():.4f}, "
            f"max={column_values.max():.4f}, "
            f"mean={column_values.mean():.4f}, "
            f"std={column_values.std():.4f}, "
            f"unique_values={len(np.unique(column_values))}"
        )

In [ ]:
# ============================================================
# STEP 18: Visualize temporal action patterns
# ============================================================

# Customizable settings
episode_to_visualize = "ep_000"

with zipfile.ZipFile(dataset_zip_path, "r") as dataset_archive:

    log_path = f"data/{episode_to_visualize}/log.npz"

    with dataset_archive.open(log_path) as log_file:
        with np.load(BytesIO(log_file.read()), allow_pickle=False) as episode_log:

            episode_timestamps = episode_log["timestamps"]
            episode_actions = episode_log["actions"]

figure, action_axes = plt.subplots(
    3, 1, figsize=(14, 8), sharex=True
)

for column_index, action_axis in enumerate(action_axes):

    action_axis.plot(
        episode_timestamps,
        episode_actions[:, column_index]
    )

    action_axis.set_ylabel(f"Action {column_index}")
    action_axis.grid(True, alpha=0.3)

action_axes[-1].set_xlabel("Timestamp (recorded units)")

figure.suptitle(
    f"Recorded Action Patterns — {episode_to_visualize}"
)

plt.tight_layout()
plt.show()

print("Episode:", episode_to_visualize)
print("Action array shape:", episode_actions.shape)
print("Timestamp array shape:", episode_timestamps.shape)

In [ ]:
# Step 20: Compare driving images with recorded actions

episode_to_visualize = "ep_000"

# Select representative points throughout the episode
sample_indices = [0, 40, 80, 120, 160, 200, 229]

with zipfile.ZipFile(dataset_zip_path, "r") as z:

    frame_paths = sorted(
        name for name in z.namelist()
        if name.startswith(
            f"data/{episode_to_visualize}/frames/"
        )
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
    )

    with z.open(f"data/{episode_to_visualize}/log.npz") as f:
        episode_log = np.load(BytesIO(f.read()))

        timestamps = episode_log["timestamps"].copy()
        actions = episode_log["actions"].copy()

    fig, axes = plt.subplots(
        2, 4, figsize=(18, 9)
    )
    axes = axes.flatten()

    for ax, idx in zip(axes, sample_indices):

        with z.open(frame_paths[idx]) as image_file:
            img = Image.open(
                BytesIO(image_file.read())
            ).convert("RGB")

        ax.imshow(img)
        ax.set_title(
            f"Frame: {idx}\n"
            f"Time: {timestamps[idx]:.2f}\n"
            f"A0: {actions[idx, 0]:.2f}, "
            f"A1: {actions[idx, 1]:.2f}, "
            f"A2: {actions[idx, 2]:.2f}"
        )
        ax.axis("off")

    axes[-1].axis("off")

    plt.suptitle(
        f"Visual Observations and Actions — {episode_to_visualize}",
        fontsize=15
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 21: Validate timestamp ordering in the smoke-test sample

timestamp_checks = []

with zipfile.ZipFile(dataset_zip_path, "r") as z:

    for episode in smoke_test_episodes:

        log_path = f"data/{episode}/log.npz"

        with z.open(log_path) as f:
            episode_log = np.load(BytesIO(f.read()))
            timestamps = episode_log["timestamps"].copy()

        differences = np.diff(timestamps)

        timestamp_checks.append({
            "episode": episode,
            "timestamp_count": len(timestamps),
            "nondecreasing": bool(np.all(differences >= 0)),
            "strictly_increasing": bool(np.all(differences > 0)),
            "duplicate_timestamps": int(np.sum(differences == 0)),
            "negative_differences": int(np.sum(differences < 0)),
            "first_timestamp": float(timestamps[0]),
            "last_timestamp": float(timestamps[-1])
        })

print("Episodes checked:", len(timestamp_checks))
print(
    "Episodes with nondecreasing timestamps:",
    sum(row["nondecreasing"] for row in timestamp_checks)
)
print(
    "Episodes with strictly increasing timestamps:",
    sum(row["strictly_increasing"] for row in timestamp_checks)
)
print(
    "Total duplicate timestamp pairs:",
    sum(row["duplicate_timestamps"] for row in timestamp_checks)
)
print(
    "Total negative timestamp differences:",
    sum(row["negative_differences"] for row in timestamp_checks)
)

for row in timestamp_checks:
    if not row["nondecreasing"]:
        print("Timestamp ordering issue:", row)

### Step 22: Confirm the available camera views
This step checks whether the dataset contains more than one camera view before any model data is built. It lists every array stored in an episode log, checks image sizes across several episodes, compares the number of images with the number of timestamps, and shows consecutive frames. A second camera would appear as a camera-related array, unusually wide images, twice as many images as timestamps, or frames that switch between the road ahead and the road behind.

In [ ]:
# STEP 22: Confirm the available camera views

check_episodes = smoke_test_episodes[:5]

with zipfile.ZipFile(dataset_zip_path, "r") as z:

    # 1. Every array stored in one episode log
    print("ARRAYS IN data/ep_000/log.npz")
    print("-" * 60)
    with z.open("data/ep_000/log.npz") as f:
        log = np.load(BytesIO(f.read()), allow_pickle=False)
        for key in log.files:
            print(f"{key:15s} shape={str(log[key].shape):12s} dtype={log[key].dtype}")

    # 2 and 3. Image size and image count vs timestamp count
    print("\nIMAGE SIZE AND COUNT CHECK")
    print("-" * 60)
    all_names = z.namelist()
    for ep in check_episodes:
        frames = sorted(
            n for n in all_names
            if n.startswith(f"data/{ep}/frames/") and n.endswith(".jpg")
        )
        with z.open(frames[0]) as f:
            width, height = Image.open(f).size
        with z.open(f"data/{ep}/log.npz") as f:
            n_times = len(np.load(BytesIO(f.read()))["timestamps"])
        print(f"{ep}: {width}x{height} px | images={len(frames)} | "
              f"timestamps={n_times} | match={len(frames) == n_times}")

    # 4. Four consecutive frames from one episode
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, i in zip(axes, range(100, 104)):
        with z.open(f"data/ep_000/frames/{i:06d}.jpg") as f:
            ax.imshow(Image.open(f).convert("RGB"))
        ax.set_title(f"ep_000, frame {i}")
        ax.axis("off")
    plt.suptitle("Consecutive frames: same view or switching views?")
    plt.tight_layout()
    plt.show()

### Step 22b: Camera View Verification Across the Full Dataset

The previous step examined a small number of episodes. This step repeats the main checks for all 297 episodes in the full dataset, before the analysis returns to the smoke-test subset. For each episode it records the folders present, the arrays stored in the log file, the size of the first image, and whether the number of images matches the number of timestamps.

In [ ]:
# Step 22b: camera check across the full dataset

all_eps = dataset_manifest["episodes"]

# group archive files by episode
files_by_ep = defaultdict(list)
for n in archive_file_names:
    parts = n.split("/")
    if len(parts) > 2 and parts[1].startswith("ep_"):
        files_by_ep[parts[1]].append(n)

folder_types = Counter()
array_sets = Counter()
image_sizes = Counter()
count_mismatch = []

with zipfile.ZipFile(dataset_zip_path) as z:
    for ep in all_eps:
        ep_files = files_by_ep[ep]

        # folders inside the episode, e.g. "frames"
        for n in ep_files:
            parts = n.split("/")
            if len(parts) > 3:
                folder_types[parts[2]] += 1

        frames = sorted(n for n in ep_files if n.endswith(".jpg"))

        with z.open(f"data/{ep}/log.npz") as f:
            log = np.load(BytesIO(f.read()))
            array_sets[tuple(sorted(log.files))] += 1
            n_ts = len(log["timestamps"])

        with z.open(frames[0]) as f:
            image_sizes[Image.open(f).size] += 1

        if len(frames) != n_ts:
            count_mismatch.append((ep, len(frames), n_ts))

print("Episodes checked:", len(all_eps))

print("\nFolders inside episodes (file counts):")
for folder, n in folder_types.items():
    print(" ", folder, n)

print("\nLog array sets found:")
for arrays, n in array_sets.items():
    print(" ", n, "episodes:", arrays)

print("\nImage sizes found:")
for size, n in image_sizes.items():
    print(" ", size, "in", n, "episodes")

print("\nEpisodes where images and timestamps don't match:", count_mismatch or "none")

In [ ]:
# files that are not frames, logs or the manifest
known = {n for n in archive_file_names
         if n.endswith(".jpg") or n.endswith("log.npz") or n == "data/manifest.json"}
others = [n for n in archive_file_names if n not in known]
print(len(others), "other files")
print(others[:20])

### Note on Additional Files

The archive contains 66 files that are not part of any episode. These are files from the data collection process rather than training data. The `data/recordings/` folder holds a `.json` and a `.log` file for each route scenario, which appear to be CARLA simulation records and route results. The remaining files, `report.json`, `_collection_state.json`, and the files in `_logs/`, record the progress and results of the collection runs.

None of these files contain camera images, so they do not change the earlier finding that the dataset includes only a front camera view. They are not needed for the smoke test. However, if the `.log` files are CARLA recorder files, they could in principle be replayed in the simulator with additional cameras attached, such as a rear camera.

### Step 23: Model Dataset Preparation

This step converts the smoke-test episodes into training examples for the action-conditioned model. Each example contains the current frame, the average action over the following five frames, and the frame five steps ahead, which corresponds to 0.25 seconds at 20 frames per second. Frames are resized to 72 by 128 pixels to reduce memory use and training time. Episodes are assigned to the training or validation set using the split field in the manifest, so that frames from the same episode never appear in both sets.

In [ ]:
# Step 23: building the model dataset

import time
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

FRAME_GAP = 5          # predict 5 frames ahead (0.25 s)
IMAGE_SIZE = (72, 128) # height, width, keeps the 16:9 shape

# train / val split from the manifest
split_of = {ep: dataset_manifest["meta"][ep]["split"] for ep in smoke_test_episodes}
train_eps = [ep for ep in smoke_test_episodes if split_of[ep] == "train"]
val_eps = [ep for ep in smoke_test_episodes if split_of[ep] == "val"]

# load the actions for every episode once
episode_actions = {}
with zipfile.ZipFile(dataset_zip_path) as z:
    for ep in smoke_test_episodes:
        with z.open(f"data/{ep}/log.npz") as f:
            episode_actions[ep] = np.load(BytesIO(f.read()))["actions"].astype(np.float32)


class DrivingPairs(Dataset):
    def __init__(self, episodes):
        self.items = [(ep, t) for ep in episodes
                      for t in range(len(episode_actions[ep]) - FRAME_GAP)]
        self.transform = T.Compose([T.Resize(IMAGE_SIZE), T.ToTensor()])
        self.zip = None

    def __len__(self):
        return len(self.items)

    def load_frame(self, ep, t):
        with self.zip.open(f"data/{ep}/frames/{t:06d}.jpg") as f:
            return self.transform(Image.open(f).convert("RGB"))

    def __getitem__(self, i):
        if self.zip is None:  # each worker opens its own copy of the zip
            self.zip = zipfile.ZipFile(dataset_zip_path)
        ep, t = self.items[i]
        action = episode_actions[ep][t:t + FRAME_GAP].mean(axis=0)
        return self.load_frame(ep, t), torch.from_numpy(action), self.load_frame(ep, t + FRAME_GAP)


train_data = DrivingPairs(train_eps)
val_data = DrivingPairs(val_eps)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

print("Train episodes:", len(train_eps), "| examples:", len(train_data))
print("Val episodes:  ", len(val_eps), "| examples:", len(val_data))

# load one batch and time it
start = time.time()
frame_now, action, frame_next = next(iter(train_loader))
print("\nOne batch loaded in", round(time.time() - start, 1), "s")
print("Current frames:", tuple(frame_now.shape))
print("Actions:       ", tuple(action.shape))
print("Future frames: ", tuple(frame_next.shape))
print("Action range:  ", action.min(0).values.numpy().round(2), "to",
      action.max(0).values.numpy().round(2))

# show one example
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(frame_now[0].permute(1, 2, 0))
axes[0].set_title("current frame")
axes[1].imshow(frame_next[0].permute(1, 2, 0))
axes[1].set_title(f"5 frames later | action {action[0].numpy().round(2)}")
for ax in axes:
    ax.axis("off")
plt.show()

**Result:**

The smoke-test data was split into 52 training episodes (24,297 examples) and 11 validation episodes (5,964 examples). Each batch contains current and future frames of size 3 x 72 x 128 and actions with three values each. The action values fall within expected ranges, and the example pair shows the scene moving forward consistently with the recorded action.

### Method Background

This model is based on two papers that train action-conditioned JEPA world models on recorded data without reinforcement learning. V-JEPA 2-AC (2025) plans actions using model predictive control. LeWorldModel (2026) trains a small model from raw images and uses the SIGReg regularizer to prevent collapse. Step 24 uses a simplified version of the LeWorldModel approach so it can run on Colab.

**References:** V-JEPA 2 (arXiv:2506.09985); LeWorldModel (arXiv:2603.19312)

### Step 24: Action-Conditioned JEPA Training

This step trains a small JEPA model on the smoke-test data. An encoder turns each frame into an embedding, and a predictor uses the current embedding and the action to predict the embedding five frames ahead. The loss combines a prediction error with SIGReg, which keeps the embeddings spread out and prevents collapse. The model trains for 300 steps and is saved for the next steps.

In [ ]:
# Step 24: action-conditioned JEPA with SIGReg (LeWorldModel style)

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

EMBED_DIM = 128
LAMBDA = 0.1      # weight of SIGReg
N_STEPS = 300
VAL_EVERY = 50
torch.manual_seed(0)


class Encoder(nn.Module):
    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        # projector with batch norm
        self.proj = nn.Sequential(nn.Linear(128, dim), nn.BatchNorm1d(dim))

    def forward(self, x):
        return self.proj(self.conv(x))


class Predictor(nn.Module):
    def __init__(self, dim=EMBED_DIM, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, dim),
        )

    def forward(self, z, a):
        return self.net(torch.cat([z, a], dim=1))


def sigreg(z, num_dirs=256, t_max=3.0, n_points=17):
    # compares random 1D projections of the embeddings with a standard normal
    n, d = z.shape
    dirs = torch.randn(d, num_dirs, device=z.device)
    dirs = dirs / dirs.norm(dim=0, keepdim=True)
    proj = z @ dirs
    t = torch.linspace(0, t_max, n_points, device=z.device)
    x = proj.unsqueeze(-1) * t
    gauss = torch.exp(-0.5 * t ** 2)
    err = (torch.cos(x).mean(0) - gauss) ** 2 + torch.sin(x).mean(0) ** 2
    stat = torch.trapz(err * gauss, t, dim=-1) * n
    return stat.mean()


encoder = Encoder().to(device)
predictor = Predictor().to(device)
optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=1e-3, weight_decay=1e-4,
)


def compute_loss(frame_now, action, frame_next):
    z_now = encoder(frame_now)
    z_next = encoder(frame_next)
    z_pred = predictor(z_now, action)
    pred_loss = F.mse_loss(z_pred, z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    return pred_loss, reg_loss, z_now


# fixed random sample of validation examples
rng = np.random.default_rng(0)
val_subset = Subset(val_data, rng.choice(len(val_data), 640, replace=False).tolist())
val_small_loader = DataLoader(val_subset, batch_size=32, num_workers=2)


def validation_loss():
    encoder.eval(); predictor.eval()
    losses = []
    with torch.no_grad():
        for f, a, n in val_small_loader:
            pred_loss, _, _ = compute_loss(f.to(device), a.to(device), n.to(device))
            losses.append(pred_loss.item())
    encoder.train(); predictor.train()
    return float(np.mean(losses))


history = {"step": [], "pred": [], "reg": [], "val": [], "spread": []}
train_iter = iter(train_loader)
start = time.time()

for step in range(1, N_STEPS + 1):
    try:
        f, a, n = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        f, a, n = next(train_iter)

    pred_loss, reg_loss, z_now = compute_loss(f.to(device), a.to(device), n.to(device))
    loss = pred_loss + LAMBDA * reg_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step == 1 or step % VAL_EVERY == 0:
        spread = z_now.std(dim=0).mean().item()
        val = validation_loss()
        for key, value in zip(history, [step, pred_loss.item(), reg_loss.item(), val, spread]):
            history[key].append(value)
        print(f"step {step:4d} | pred {pred_loss.item():.4f} | sigreg {reg_loss.item():.3f} | "
              f"val {val:.4f} | spread {spread:.3f} | {time.time() - start:.0f}s")

# save the model for the next steps
torch.save({"encoder": encoder.state_dict(), "predictor": predictor.state_dict()},
           "/content/jepa_step24.pt")
print("\nModel saved to /content/jepa_step24.pt")

# training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
axes[0].plot(history["step"], history["pred"], marker="o", label="train")
axes[0].plot(history["step"], history["val"], marker="o", label="val")
axes[0].set_title("Prediction loss"); axes[0].legend()
axes[1].plot(history["step"], history["reg"], marker="o")
axes[1].set_title("SIGReg")
axes[2].plot(history["step"], history["spread"], marker="o")
axes[2].set_title("Embedding spread (near 0 = collapse)")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout()
plt.show()

**Result:** Training completed in about six minutes on the GPU without errors. The embedding spread increased from 0.40 to about 0.99 and remained stable, so no representation collapse occurred. SIGReg decreased from 7.8 to about 3.5. The training prediction loss fell from 0.20 to between 0.015 and 0.04, and the validation loss decreased steadily from 0.21 at step 50 to 0.053 at step 300. The validation loss at step 1 is not meaningful, because the embeddings were not yet spread out at that point. The gap between training and validation loss suggests some overfitting, which is expected for a short training run on a small dataset.

### Step 25: Baseline and Action Checks

This step checks whether the trained model has learned useful predictions and whether it relies on the actions. The model's prediction error on validation examples is compared with two simple baselines: assuming the scene does not change, and predicting the average future embedding. The model is also evaluated with shuffled actions and with all actions set to zero. A lower error than both baselines indicates that the model has learned how the scene changes, and a higher error with shuffled or zero actions indicates that the model uses the action information.

In [ ]:
# Step 25: baselines and action checks

encoder.eval()
predictor.eval()
torch.manual_seed(0)

errors = {"model": [], "no change": [], "average": [], "shuffled actions": [], "zero actions": []}


def per_example_error(pred, target):
    return ((pred - target) ** 2).mean(dim=1)


with torch.no_grad():
    for f, a, n in val_small_loader:
        f, a, n = f.to(device), a.to(device), n.to(device)
        z_now, z_next = encoder(f), encoder(n)

        errors["model"].append(per_example_error(predictor(z_now, a), z_next))
        errors["no change"].append(per_example_error(z_now, z_next))
        avg = z_next.mean(dim=0, keepdim=True).expand_as(z_next)
        errors["average"].append(per_example_error(avg, z_next))
        shuffled = a[torch.randperm(len(a), device=device)]
        errors["shuffled actions"].append(per_example_error(predictor(z_now, shuffled), z_next))
        errors["zero actions"].append(per_example_error(predictor(z_now, torch.zeros_like(a)), z_next))

errors = {k: torch.cat(v).cpu() for k, v in errors.items()}
mean_err = {k: v.mean().item() for k, v in errors.items()}

print("Validation examples:", len(errors["model"]))
print("\nMean prediction error (lower is better):")
for k, v in mean_err.items():
    print(f"  {k:17s} {v:.4f}   ({v / mean_err['model']:.2f}x the model)")

beat = (errors["model"] < errors["no change"]).float().mean().item()
print(f"\nModel beats the no-change guess on {beat:.0%} of examples")

plt.figure(figsize=(8, 3.5))
plt.bar(list(mean_err.keys()), list(mean_err.values()))
plt.ylabel("mean prediction error")
plt.title("Model vs baselines and action checks (validation)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Interpretation:**

The model's validation error (0.053) is much lower than the average-embedding baseline (0.964), so the embeddings capture scene-specific information. However, the model performs worse than the no-change baseline (0.011) and does not beat it on any validation example. The error is also unchanged when actions are shuffled or set to zero, which shows that the model does not yet use the action information. Likely causes are that the predictor must rebuild the future embedding from scratch, the encoder discards spatial layout through global pooling, and the short training run of 300 steps. These issues are addressed in the next step.

### Step 26: Improved Action-Conditioned Model

This step revises the model based on the results of Step 25. The predictor now outputs the change between the current and future embeddings and is initialised to predict no change, so it starts at the no-change baseline. The encoder keeps a coarse 2 by 4 spatial layout instead of averaging the whole image, which preserves information about where objects are and how they move. Actions are passed through a small network before being combined with the embedding. The model is trained for 1,000 steps, and the no-change and shuffled-action comparisons are tracked during training.

In [ ]:
# Step 26: improved model

class EncoderV2(nn.Module):
    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 4)), nn.Flatten(),  # keeps a rough 2x4 layout
        )
        self.proj = nn.Sequential(nn.Linear(128 * 8, dim), nn.BatchNorm1d(dim))

    def forward(self, x):
        return self.proj(self.conv(x))


class PredictorV2(nn.Module):
    def __init__(self, dim=EMBED_DIM, action_dim=3):
        super().__init__()
        self.action_net = nn.Sequential(nn.Linear(action_dim, 64), nn.ReLU(), nn.Linear(64, 64))
        self.net = nn.Sequential(
            nn.Linear(dim + 64, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, dim),
        )
        # last layer starts at zero, so the first prediction is "no change"
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, z, a):
        return z + self.net(torch.cat([z, self.action_net(a)], dim=1))


torch.manual_seed(0)
encoder2 = EncoderV2().to(device)
predictor2 = PredictorV2().to(device)
optimizer2 = torch.optim.AdamW(
    list(encoder2.parameters()) + list(predictor2.parameters()),
    lr=1e-3, weight_decay=1e-4,
)

N_STEPS_V2 = 1000
EVAL_EVERY = 100


def evaluate_v2():
    encoder2.eval(); predictor2.eval()
    model_err, still_err, shuffle_err = [], [], []
    with torch.no_grad():
        for f, a, n in val_small_loader:
            f, a, n = f.to(device), a.to(device), n.to(device)
            z_now, z_next = encoder2(f), encoder2(n)
            model_err.append(((predictor2(z_now, a) - z_next) ** 2).mean(dim=1))
            still_err.append(((z_now - z_next) ** 2).mean(dim=1))
            shuffled = a[torch.randperm(len(a), device=device)]
            shuffle_err.append(((predictor2(z_now, shuffled) - z_next) ** 2).mean(dim=1))
    encoder2.train(); predictor2.train()
    model_err, still_err, shuffle_err = map(torch.cat, (model_err, still_err, shuffle_err))
    beat = (model_err < still_err).float().mean().item()
    return model_err.mean().item(), still_err.mean().item(), shuffle_err.mean().item(), beat


log = {"step": [], "model": [], "no_change": [], "shuffled": [], "beat": [], "spread": []}
train_iter = iter(train_loader)
start = time.time()

for step in range(1, N_STEPS_V2 + 1):
    try:
        f, a, n = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        f, a, n = next(train_iter)
    f, a, n = f.to(device), a.to(device), n.to(device)

    z_now, z_next = encoder2(f), encoder2(n)
    pred_loss = F.mse_loss(predictor2(z_now, a), z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    loss = pred_loss + LAMBDA * reg_loss

    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()

    if step == 1 or step % EVAL_EVERY == 0:
        m, s, sh, beat = evaluate_v2()
        spread = z_now.std(dim=0).mean().item()
        for key, value in zip(log, [step, m, s, sh, beat, spread]):
            log[key].append(value)
        print(f"step {step:4d} | model {m:.4f} | no change {s:.4f} | shuffled {sh:.4f} | "
              f"beats no-change {beat:.0%} | spread {spread:.2f} | {time.time() - start:.0f}s")

torch.save({"encoder": encoder2.state_dict(), "predictor": predictor2.state_dict()},
           "/content/jepa_step26.pt")
print("\nModel saved to /content/jepa_step26.pt")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(log["step"], log["model"], marker="o", label="model")
axes[0].plot(log["step"], log["no_change"], marker="o", label="no change")
axes[0].plot(log["step"], log["shuffled"], marker="o", label="shuffled actions")
axes[0].set_title("Validation error"); axes[0].set_xlabel("step"); axes[0].legend()
axes[1].plot(log["step"], log["beat"], marker="o")
axes[1].set_title("Share of examples where the model beats no-change")
axes[1].set_xlabel("step")
plt.tight_layout()
plt.show()

**Result:** The revised model trained for 1,000 steps without collapse, with the embedding spread stable at about 0.96. At the final step, the model's validation error (0.0279) was 19 percent lower than the no-change baseline (0.0344), and the model outperformed the baseline on 53 percent of examples, compared with 0 percent in Step 24. From step 400 onward, shuffling the actions increased the error at every evaluation, and at the final step the shuffled-action error (0.0378) was 26 percent higher than the model's error. This indicates that the model now uses the action information. The results fluctuate between evaluations, and the model was briefly worse than the baseline at steps 600 and 800, so the improvement is present but not yet stable. These results are based on 640 validation examples from a single training run.

### Step 27: Frame Caching for Faster Training

This step resizes all frames from the smoke-test episodes to 72 by 128 pixels once and stores them in memory, instead of decoding the full-size images from the archive at every training step. The resized frames are saved to Google Drive so that later sessions can load them directly. The frames are then moved to the GPU, and training examples are drawn from there, which greatly reduces training time.

In [ ]:
# Step 27: cache resized frames in memory and on Drive

from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/DriveJEPA"
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_PATH = f"{CACHE_DIR}/smoke_frames_72x128.npz"
H, W = 72, 128


def load_small(z, path):
    with z.open(path) as f:
        img = Image.open(f)
        img.draft("RGB", (W * 2, H * 2))  # faster decode at reduced size
        return np.asarray(img.convert("RGB").resize((W, H), Image.BILINEAR))


if os.path.exists(CACHE_PATH):
    saved = np.load(CACHE_PATH)
    frame_cache = {ep: saved[ep] for ep in smoke_test_episodes}
    print("Loaded frames from Drive")
else:
    frame_cache = {}
    start = time.time()
    with zipfile.ZipFile(dataset_zip_path) as z:
        for i, ep in enumerate(smoke_test_episodes, 1):
            n = len(episode_actions[ep])
            frame_cache[ep] = np.stack(
                [load_small(z, f"data/{ep}/frames/{t:06d}.jpg") for t in range(n)]
            )
            if i % 10 == 0:
                print(f"  {i}/{len(smoke_test_episodes)} episodes done, {time.time() - start:.0f}s")
    np.savez(CACHE_PATH, **frame_cache)
    print("Saved frames to", CACHE_PATH)

# put all frames on the GPU
offsets, pos = {}, 0
for ep in smoke_test_episodes:
    offsets[ep] = pos
    pos += len(frame_cache[ep])

all_frames = torch.from_numpy(
    np.concatenate([frame_cache[ep] for ep in smoke_test_episodes])
).permute(0, 3, 1, 2).contiguous().to(device)   # kept as uint8 to save memory

print(f"Frames on GPU: {tuple(all_frames.shape)}, {all_frames.numel() / 1e9:.2f} GB")


def build_index(episodes, gap):
    # positions of current and future frames, plus the average action between them
    now, nxt, act = [], [], []
    for ep in episodes:
        o, acts = offsets[ep], episode_actions[ep]
        for t in range(len(acts) - gap):
            now.append(o + t)
            nxt.append(o + t + gap)
            act.append(acts[t:t + gap].mean(axis=0))
    return (torch.tensor(now, device=device),
            torch.tensor(nxt, device=device),
            torch.tensor(np.array(act), device=device))


def get_batch(index, ids):
    now, nxt, act = index
    return all_frames[now[ids]].float() / 255, act[ids], all_frames[nxt[ids]].float() / 255


train_index = build_index(train_eps, FRAME_GAP)
val_index = build_index(val_eps, FRAME_GAP)
print("Train examples:", len(train_index[0]), "| Val examples:", len(val_index[0]))

# speed check: 100 random batches of 64
start = time.time()
for _ in range(100):
    ids = torch.randint(0, len(train_index[0]), (64,), device=device)
    f, a, n = get_batch(train_index, ids)
torch.cuda.synchronize()
print(f"100 batches of 64 in {time.time() - start:.2f}s")

# quick visual check of a cached frame
plt.figure(figsize=(5, 3))
plt.imshow(frame_cache[smoke_test_episodes[0]][100])
plt.title(f"Cached frame: {smoke_test_episodes[0]}, frame 100")
plt.axis("off")
plt.show()

**Result:**

All frames from the 63 smoke-test episodes were resized and saved to Google Drive in about 2.7 minutes. The cached frames occupy 0.85 GB on the GPU, and the training and validation splits are unchanged at 24,297 and 5,964 examples. Loading 100 batches of 64 examples now takes 0.1 seconds, compared with about one second per batch of 32 previously. The sample frame confirms that the resized images are correct.

### Step 28: Full Validation Evaluation

This step evaluates the Step 26 model, referred to as v1, on all 5,964 validation examples using the cached frames. The model is compared with the no-change baseline, with actions shuffled across the full validation set, and with all actions set to zero. Whole validation episodes are resampled 1,000 times to give a 95 percent interval for each improvement, and results are grouped by weather condition. The evaluation code defined here is reused for later model versions, and the results are collected in a single comparison table. Because the validation set contains only 11 episodes, the intervals are wide and the weather breakdown is indicative only.

In [ ]:
# Step 28: full validation evaluation (v1 = Step 26 model)

import shutil
import pandas as pd

# keep a copy of the v1 model on Drive
shutil.copy("/content/jepa_step26.pt", f"{CACHE_DIR}/jepa_v1_step26.pt")


def example_episodes(episodes, gap):
    # episode name for every example, in the same order as build_index
    return [ep for ep in episodes for _ in range(len(episode_actions[ep]) - gap)]


def evaluate_full(enc, pred, index, example_eps, seed=0):
    enc.eval(); pred.eval()
    n = len(index[0])
    z_now, z_next, acts = [], [], []
    with torch.no_grad():
        for s in range(0, n, 512):
            ids = torch.arange(s, min(s + 512, n), device=device)
            f, a, nx = get_batch(index, ids)
            z_now.append(enc(f)); z_next.append(enc(nx)); acts.append(a)
        z_now, z_next, acts = torch.cat(z_now), torch.cat(z_next), torch.cat(acts)

        def err(p):
            return ((p - z_next) ** 2).mean(dim=1).cpu().numpy()

        g = torch.Generator(device=device).manual_seed(seed)
        shuffled = acts[torch.randperm(n, device=device, generator=g)]
        df = pd.DataFrame({
            "episode": example_eps,
            "model": err(pred(z_now, acts)),
            "no_change": err(z_now),
            "shuffled": err(pred(z_now, shuffled)),
            "zero": err(pred(z_now, torch.zeros_like(acts))),
        })
    enc.train(); pred.train()
    return df


def summarize(df, name):
    m = df[["model", "no_change", "shuffled", "zero"]].mean()
    per_ep = df.groupby("episode")[["model", "no_change", "shuffled"]].mean()
    rng = np.random.default_rng(0)
    gain, shuf = [], []
    for _ in range(1000):
        s = per_ep.loc[rng.choice(per_ep.index, len(per_ep))]
        gain.append(1 - s["model"].mean() / s["no_change"].mean())
        shuf.append(s["shuffled"].mean() / s["model"].mean() - 1)
    return {
        "version": name,
        "model": round(m["model"], 4),
        "no_change": round(m["no_change"], 4),
        "shuffled": round(m["shuffled"], 4),
        "gain_vs_no_change": f"{1 - m['model'] / m['no_change']:.1%}",
        "gain_95%": f"{np.percentile(gain, 2.5):.1%} to {np.percentile(gain, 97.5):.1%}",
        "shuffle_increase": f"{m['shuffled'] / m['model'] - 1:.1%}",
        "shuffle_95%": f"{np.percentile(shuf, 2.5):.1%} to {np.percentile(shuf, 97.5):.1%}",
        "beats_no_change": f"{(df['model'] < df['no_change']).mean():.0%}",
    }


val_example_eps = example_episodes(val_eps, FRAME_GAP)

start = time.time()
v1_results = evaluate_full(encoder2, predictor2, val_index, val_example_eps)
print(f"Evaluated {len(v1_results)} validation examples in {time.time() - start:.1f}s\n")

results_table = [summarize(v1_results, "v1 (Step 26)")]
display(pd.DataFrame(results_table))

# by weather
v1_results["weather"] = v1_results["episode"].map(lambda ep: dataset_manifest["meta"][ep]["weather"])
by_weather = v1_results.groupby("weather").agg(
    episodes=("episode", "nunique"),
    examples=("model", "size"),
    model=("model", "mean"),
    no_change=("no_change", "mean"),
    shuffled=("shuffled", "mean"),
)
by_weather["gain_vs_no_change"] = 1 - by_weather["model"] / by_weather["no_change"]
display(by_weather.round(4))

**Result:**

On all 5,964 validation examples, the v1 model's error (0.0354) was 17.1 percent lower than the no-change baseline (0.0427), with a 95 percent interval of 8.4 to 23.6 percent. Shuffling the actions increased the error by 40.8 percent (95 percent interval 26.3 to 61.0 percent), and the shuffled-action error was higher than the no-change baseline, which shows that the model depends on the action information. The model outperformed the baseline on 51 percent of examples, so the average gain comes mainly from examples where the improvement is large. By weather, the largest gains were for clear night, fog, clear sunset, and cloudy noon, while the model performed close to or slightly worse than the baseline in soft and hard rain. Each weather group contains only one to three episodes, and five of the twelve weather types are not represented in the validation set, so the weather results are indicative only.

### Step 29: Improved Training Recipe (v2)

This step trains a new model with the same architecture as v1 but an improved training recipe, so that any change in results can be attributed to the training procedure. The model is trained for 5,000 steps instead of 1,000, with a batch size of 128 instead of 32. The learning rate increases during a short warm-up and then decreases gradually, and gradient clipping limits the size of each update. Training uses the cached frames. Validation results are recorded every 500 steps for monitoring only, and the final model is evaluated with the same full validation procedure as v1.

In [ ]:
# Step 29: v2 = same model as v1, better training recipe

import math

torch.manual_seed(0)
enc_v2 = EncoderV2().to(device)
pred_v2 = PredictorV2().to(device)
params_v2 = list(enc_v2.parameters()) + list(pred_v2.parameters())
opt_v2 = torch.optim.AdamW(params_v2, lr=1e-3, weight_decay=1e-4)

STEPS = 5000
BATCH = 128
WARMUP = 250
EVAL_EVERY = 500


def lr_factor(step):
    # short warm-up, then a smooth decrease to zero
    if step < WARMUP:
        return (step + 1) / WARMUP
    progress = (step - WARMUP) / (STEPS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))


sched_v2 = torch.optim.lr_scheduler.LambdaLR(opt_v2, lr_factor)

log_v2 = {"step": [], "model": [], "no_change": [], "shuffled": [], "beat": [], "spread": []}
n_train = len(train_index[0])
start = time.time()

for step in range(1, STEPS + 1):
    ids = torch.randint(0, n_train, (BATCH,), device=device)
    f, a, n = get_batch(train_index, ids)

    z_now, z_next = enc_v2(f), enc_v2(n)
    pred_loss = F.mse_loss(pred_v2(z_now, a), z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    loss = pred_loss + LAMBDA * reg_loss

    opt_v2.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_v2, 1.0)
    opt_v2.step()
    sched_v2.step()

    if step % EVAL_EVERY == 0:
        df = evaluate_full(enc_v2, pred_v2, val_index, val_example_eps)
        m, s, sh = df["model"].mean(), df["no_change"].mean(), df["shuffled"].mean()
        beat = (df["model"] < df["no_change"]).mean()
        spread = z_now.std(dim=0).mean().item()
        for key, value in zip(log_v2, [step, m, s, sh, beat, spread]):
            log_v2[key].append(value)
        print(f"step {step:5d} | model {m:.4f} | no change {s:.4f} | shuffled {sh:.4f} | "
              f"beats no-change {beat:.0%} | spread {spread:.2f} | {time.time() - start:.0f}s")

# save and evaluate the final model
torch.save({"encoder": enc_v2.state_dict(), "predictor": pred_v2.state_dict()},
           f"{CACHE_DIR}/jepa_v2_step29.pt")
print("\nModel saved to Drive")

v2_results = evaluate_full(enc_v2, pred_v2, val_index, val_example_eps)
results_table.append(summarize(v2_results, "v2 (Step 29)"))
display(pd.DataFrame(results_table))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(log_v2["step"], log_v2["model"], marker="o", label="model")
axes[0].plot(log_v2["step"], log_v2["no_change"], marker="o", label="no change")
axes[0].plot(log_v2["step"], log_v2["shuffled"], marker="o", label="shuffled actions")
axes[0].set_title("Validation error (v2)"); axes[0].set_xlabel("step"); axes[0].legend()
axes[1].plot(log_v2["step"], log_v2["beat"], marker="o")
axes[1].set_title("Share of examples where v2 beats no-change"); axes[1].set_xlabel("step")
plt.tight_layout()
plt.show()

In [ ]:
print(log_v2["step"])
print([round(x, 3) for x in log_v2["model"]])
print([round(x, 3) for x in log_v2["no_change"]])
print([f"{x:.0%}" for x in log_v2["beat"]])

In [ ]:
display(pd.DataFrame(results_table))

**Result:** The v2 model performed worse than v1. Its validation error was 18.4 percent higher than the no-change baseline (95 percent interval from 41.4 percent worse to 8.1 percent better), and it outperformed the baseline on only 22 percent of examples, compared with 51 percent for v1. Shuffling the actions still increased the error by 28.9 percent (95 percent interval 17.9 to 35.1 percent), so the model continued to use the action information. During training, the advantage over the baseline declined steadily after the first 500 steps. With 5,000 steps at a batch size of 128, the model passed over the 52 training episodes about 26 times, compared with about 1.3 times for v1, which indicates overfitting. Longer training alone does not improve the model when the amount of training data is this small.

### Step 30: Reducing Overfitting (v3)

This step trains a new model designed to reduce the overfitting seen in v2. Three types of data augmentation are applied during training. Half of each batch is flipped horizontally, with the sign of the steering action reversed so that a left turn becomes a right turn. Frames are also shifted by up to four pixels and their brightness is varied by up to 20 percent, with the same change applied to the current and future frames. The predictor uses 10 percent dropout, as in LeWorldModel, and the weight decay is increased to 0.05. Training is limited to about four passes over the training data. The flip augmentation assumes that the first action column is steering, which is consistent with its symmetric range and with the dataset's processing log.

In [ ]:
# Step 30: v3 = augmentation + dropout + fewer passes

class PredictorV3(nn.Module):
    def __init__(self, dim=EMBED_DIM, action_dim=3, dropout=0.1):
        super().__init__()
        self.action_net = nn.Sequential(nn.Linear(action_dim, 64), nn.ReLU(), nn.Linear(64, 64))
        self.net = nn.Sequential(
            nn.Linear(dim + 64, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, dim),
        )
        nn.init.zeros_(self.net[-1].weight)  # starts as "no change"
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, z, a):
        return z + self.net(torch.cat([z, self.action_net(a)], dim=1))


def augment(f, a, n, max_shift=4):
    a = a.clone()
    # flip half the batch left-right and reverse the steering sign
    flip = torch.rand(len(f), device=device) < 0.5
    f = torch.where(flip[:, None, None, None], f.flip(-1), f)
    n = torch.where(flip[:, None, None, None], n.flip(-1), n)
    a[:, 0] = torch.where(flip, -a[:, 0], a[:, 0])
    # same small shift for current and future frames
    dx, dy = np.random.randint(-max_shift, max_shift + 1, size=2)
    f = torch.roll(f, shifts=(int(dy), int(dx)), dims=(2, 3))
    n = torch.roll(n, shifts=(int(dy), int(dx)), dims=(2, 3))
    # same brightness change for both frames
    scale = torch.empty(len(f), 1, 1, 1, device=device).uniform_(0.8, 1.2)
    return (f * scale).clamp(0, 1), a, (n * scale).clamp(0, 1)


torch.manual_seed(0)
np.random.seed(0)
enc_v3 = EncoderV2().to(device)
pred_v3 = PredictorV3().to(device)
params_v3 = list(enc_v3.parameters()) + list(pred_v3.parameters())
opt_v3 = torch.optim.AdamW(params_v3, lr=1e-3, weight_decay=0.05)

STEPS = 1500
BATCH = 64
WARMUP = 100
EVAL_EVERY = 250
sched_v3 = torch.optim.lr_scheduler.LambdaLR(opt_v3, lr_factor)  # same schedule shape as v2

print(f"Passes over training data: {STEPS * BATCH / n_train:.1f}")

log_v3 = {"step": [], "model": [], "no_change": [], "shuffled": [], "beat": [], "spread": []}
start = time.time()

for step in range(1, STEPS + 1):
    ids = torch.randint(0, n_train, (BATCH,), device=device)
    f, a, n = augment(*get_batch(train_index, ids))

    z_now, z_next = enc_v3(f), enc_v3(n)
    pred_loss = F.mse_loss(pred_v3(z_now, a), z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    loss = pred_loss + LAMBDA * reg_loss

    opt_v3.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_v3, 1.0)
    opt_v3.step()
    sched_v3.step()

    if step % EVAL_EVERY == 0:
        df = evaluate_full(enc_v3, pred_v3, val_index, val_example_eps)
        m, s, sh = df["model"].mean(), df["no_change"].mean(), df["shuffled"].mean()
        beat = (df["model"] < df["no_change"]).mean()
        spread = z_now.std(dim=0).mean().item()
        for key, value in zip(log_v3, [step, m, s, sh, beat, spread]):
            log_v3[key].append(value)
        print(f"step {step:5d} | model {m:.4f} | no change {s:.4f} | shuffled {sh:.4f} | "
              f"beats no-change {beat:.0%} | spread {spread:.2f} | {time.time() - start:.0f}s")

torch.save({"encoder": enc_v3.state_dict(), "predictor": pred_v3.state_dict()},
           f"{CACHE_DIR}/jepa_v3_step30.pt")
print("\nModel saved to Drive")

v3_results = evaluate_full(enc_v3, pred_v3, val_index, val_example_eps)
results_table.append(summarize(v3_results, "v3 (Step 30)"))
display(pd.DataFrame(results_table))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(log_v3["step"], log_v3["model"], marker="o", label="model")
axes[0].plot(log_v3["step"], log_v3["no_change"], marker="o", label="no change")
axes[0].plot(log_v3["step"], log_v3["shuffled"], marker="o", label="shuffled actions")
axes[0].set_title("Validation error (v3)"); axes[0].set_xlabel("step"); axes[0].legend()
axes[1].plot(log_v3["step"], log_v3["beat"], marker="o")
axes[1].set_title("Share of examples where v3 beats no-change"); axes[1].set_xlabel("step")
plt.tight_layout()
plt.show()

**Result:**

 The v3 model trained in 43 seconds without overfitting. It outperformed the no-change baseline at every evaluation, and the share of examples where it beat the baseline rose from 40 percent to about 54 percent and then remained stable. On the full validation set, its error was 16.8 percent lower than the baseline (95 percent interval 10.7 to 23.2 percent), and it outperformed the baseline on 54 percent of examples. Shuffling the actions increased the error by 30.6 percent (95 percent interval 20.2 to 41.0 percent). Compared with v1, the average gain is similar, but the lower bound of the interval is higher and the training behaviour is more stable, while v2's overfitting was removed. Each result comes from a single training run, so small differences between versions should be interpreted with caution.

### Step 31: Training on All Training Episodes (v4)

This step tests whether more training data improves the model. The v4 model uses the same architecture and training recipe as v3, but it is trained on every episode marked as training data in the full dataset, rather than only the 52 training episodes in the smoke-test subset. For comparison with earlier versions, v4 is evaluated on the same 11 validation episodes. It is also evaluated on all validation episodes in the full dataset, which provides a larger test set covering more weather conditions. No validation episode is used for training, and the near-duplicate episode pairs found in the manifest do not cross between the training and validation splits. As a consistency check, v3 is re-evaluated after the frames are reloaded and should give the same results as before.

In [ ]:
# Step 31: v4 = v3 recipe trained on all training episodes

SAVE_FULL_CACHE = False   # set True to also save the full cache to Drive (about 3.5 GB)

all_eps = dataset_manifest["episodes"]
train_eps_all = [ep for ep in all_eps if dataset_manifest["meta"][ep]["split"] == "train"]
val_eps_all = [ep for ep in all_eps if dataset_manifest["meta"][ep]["split"] == "val"]
print("Training episodes:", len(train_eps_all), "| Validation episodes:", len(val_eps_all))

# 1. cache frames and actions for the episodes not loaded yet
needed = [ep for ep in all_eps if ep not in frame_cache]
print("Episodes still to cache:", len(needed))
start = time.time()
with zipfile.ZipFile(dataset_zip_path) as z:
    for i, ep in enumerate(needed, 1):
        with z.open(f"data/{ep}/log.npz") as f:
            episode_actions[ep] = np.load(BytesIO(f.read()))["actions"].astype(np.float32)
        n = len(episode_actions[ep])
        frame_cache[ep] = np.stack(
            [load_small(z, f"data/{ep}/frames/{t:06d}.jpg") for t in range(n)]
        )
        if i % 25 == 0:
            print(f"  {i}/{len(needed)} episodes cached, {time.time() - start:.0f}s")
print(f"Caching done in {time.time() - start:.0f}s")

if SAVE_FULL_CACHE:
    np.savez(f"{CACHE_DIR}/all_frames_72x128.npz", **frame_cache)
    print("Full cache saved to Drive")

# 2. move every frame to the GPU, one episode at a time to save memory
del all_frames
torch.cuda.empty_cache()
total = sum(len(frame_cache[ep]) for ep in all_eps)
all_frames = torch.empty((total, 3, H, W), dtype=torch.uint8, device=device)
offsets, pos = {}, 0
for ep in all_eps:
    arr = torch.from_numpy(frame_cache[ep]).permute(0, 3, 1, 2)
    all_frames[pos:pos + len(arr)] = arr.to(device)
    offsets[ep] = pos
    pos += len(arr)
print(f"Frames on GPU: {tuple(all_frames.shape)}, {all_frames.numel() / 1e9:.2f} GB")

# rebuild indexes, since frame positions changed
train_index_all = build_index(train_eps_all, FRAME_GAP)
val_index = build_index(val_eps, FRAME_GAP)            # same 11 episodes as before
val_index_all = build_index(val_eps_all, FRAME_GAP)
val_example_eps_all = example_episodes(val_eps_all, FRAME_GAP)
n_train_all = len(train_index_all[0])
print("Train examples:", n_train_all, "| Val examples (11 eps):", len(val_index[0]),
      "| Val examples (all val eps):", len(val_index_all[0]))

# 3. consistency check: v3 should match its earlier result
check = summarize(evaluate_full(enc_v3, pred_v3, val_index, val_example_eps), "v3 re-check")
print("v3 re-check gain vs no change:", check["gain_vs_no_change"], "(earlier: 16.8%)")

# 4. train v4 with the v3 recipe for about 4 passes
torch.manual_seed(0)
np.random.seed(0)
enc_v4 = EncoderV2().to(device)
pred_v4 = PredictorV3().to(device)
params_v4 = list(enc_v4.parameters()) + list(pred_v4.parameters())
opt_v4 = torch.optim.AdamW(params_v4, lr=1e-3, weight_decay=0.05)

BATCH = 64
STEPS = int(4 * n_train_all / BATCH)
WARMUP = 250
EVAL_EVERY = 1000
sched_v4 = torch.optim.lr_scheduler.LambdaLR(opt_v4, lr_factor)
print(f"\nTraining v4 for {STEPS} steps (about 4 passes)")

log_v4 = {"step": [], "model": [], "no_change": [], "shuffled": [], "beat": []}
start = time.time()

for step in range(1, STEPS + 1):
    ids = torch.randint(0, n_train_all, (BATCH,), device=device)
    f, a, n = augment(*get_batch(train_index_all, ids))

    z_now, z_next = enc_v4(f), enc_v4(n)
    pred_loss = F.mse_loss(pred_v4(z_now, a), z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    loss = pred_loss + LAMBDA * reg_loss

    opt_v4.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_v4, 1.0)
    opt_v4.step()
    sched_v4.step()

    if step % EVAL_EVERY == 0 or step == STEPS:
        df = evaluate_full(enc_v4, pred_v4, val_index, val_example_eps)
        m, s, sh = df["model"].mean(), df["no_change"].mean(), df["shuffled"].mean()
        beat = (df["model"] < df["no_change"]).mean()
        for key, value in zip(log_v4, [step, m, s, sh, beat]):
            log_v4[key].append(value)
        print(f"step {step:5d} | model {m:.4f} | no change {s:.4f} | shuffled {sh:.4f} | "
              f"beats no-change {beat:.0%} | {time.time() - start:.0f}s")

torch.save({"encoder": enc_v4.state_dict(), "predictor": pred_v4.state_dict()},
           f"{CACHE_DIR}/jepa_v4_step31.pt")
print("\nModel saved to Drive")

# 5. results on the same 11 validation episodes
v4_results = evaluate_full(enc_v4, pred_v4, val_index, val_example_eps)
results_table.append(summarize(v4_results, "v4 (Step 31)"))
print("\nSame 11 validation episodes as v1 to v3:")
print(pd.DataFrame(results_table).to_string(index=False))

# results on all validation episodes, v3 vs v4
full_rows = [
    summarize(evaluate_full(enc_v3, pred_v3, val_index_all, val_example_eps_all), "v3 (52 train eps)"),
    summarize(evaluate_full(enc_v4, pred_v4, val_index_all, val_example_eps_all), "v4 (all train eps)"),
]
print(f"\nAll {len(val_eps_all)} validation episodes:")
print(pd.DataFrame(full_rows).to_string(index=False))

# weather breakdown for v4 on all validation episodes
v4_full = evaluate_full(enc_v4, pred_v4, val_index_all, val_example_eps_all)
v4_full["weather"] = v4_full["episode"].map(lambda ep: dataset_manifest["meta"][ep]["weather"])
by_weather_v4 = v4_full.groupby("weather").agg(
    episodes=("episode", "nunique"),
    examples=("model", "size"),
    model=("model", "mean"),
    no_change=("no_change", "mean"),
    shuffled=("shuffled", "mean"),
)
by_weather_v4["gain_vs_no_change"] = 1 - by_weather_v4["model"] / by_weather_v4["no_change"]
print("\nv4 by weather (all validation episodes):")
print(by_weather_v4.round(4).to_string())

In [ ]:
# check whether v4 finished
print("Steps logged:", log_v4["step"])
print("Versions in table:", [r["version"] for r in results_table])
print("v4 saved to Drive:", os.path.exists(f"{CACHE_DIR}/jepa_v4_step31.pt"))

In [ ]:
# show v4 results again without retraining
print("Same 11 validation episodes as v1 to v3:")
print(pd.DataFrame(results_table).to_string(index=False))

full_rows = [
    summarize(evaluate_full(enc_v3, pred_v3, val_index_all, val_example_eps_all), "v3 (52 train eps)"),
    summarize(evaluate_full(enc_v4, pred_v4, val_index_all, val_example_eps_all), "v4 (all train eps)"),
]
print(f"\nAll {len(val_eps_all)} validation episodes:")
print(pd.DataFrame(full_rows).to_string(index=False))

v4_full = evaluate_full(enc_v4, pred_v4, val_index_all, val_example_eps_all)
v4_full["weather"] = v4_full["episode"].map(lambda ep: dataset_manifest["meta"][ep]["weather"])
by_weather_v4 = v4_full.groupby("weather").agg(
    episodes=("episode", "nunique"),
    examples=("model", "size"),
    model=("model", "mean"),
    no_change=("no_change", "mean"),
    shuffled=("shuffled", "mean"),
)
by_weather_v4["gain_vs_no_change"] = 1 - by_weather_v4["model"] / by_weather_v4["no_change"]
print("\nv4 by weather (all validation episodes):")
print(by_weather_v4.round(4).to_string())

In [ ]:
for i in range(len(log_v4["step"])):
    print(f"step {log_v4['step'][i]:5d} | model {log_v4['model'][i]:.4f} | "
          f"no change {log_v4['no_change'][i]:.4f} | shuffled {log_v4['shuffled'][i]:.4f} | "
          f"beats no-change {log_v4['beat'][i]:.0%}")

**Result:**

 Training on all 250 training episodes improved the model. During training, v4 outperformed the no-change baseline at every evaluation, and the share of validation examples where it beat the baseline rose from 52 percent at step 1,000 to 63 percent at the final step, with no sign of overfitting. On the same 11 validation episodes used for earlier versions, v4 reduced the error by 18.2 percent compared with the no-change baseline, outperformed the baseline on 63 percent of examples (54 percent for v3), and showed a 38.7 percent increase in error when actions were shuffled. On all 47 validation episodes, v4 reduced the error by 20.7 percent (95 percent interval 16.4 to 25.9 percent), compared with 18.5 percent for v3, and shuffling the actions increased the error by 41.6 percent. The improvement over v3 is consistent across all measures and both validation sets, although the intervals overlap. The model outperformed the baseline in all 11 weather conditions present in the validation set, with gains between 11 and 31 percent. The earlier result that rain conditions were the most difficult does not hold with the larger validation set, which suggests it was caused by the small number of episodes. The 47-episode validation set is used as the main benchmark from this point on.

### Step 32: Action Response Test and Closest-Frame Visualization

This step checks whether the v4 model responds sensibly to different actions, using all 47 validation episodes. For moments where the recorded steering was clearly positive or negative, the model's prediction is compared under a positive and a negative steering value, and the test counts how often the steering that matches the recording gives the lower error. The same test is repeated for braking and accelerating. Random guessing would score 50 percent. Because the model predicts embeddings rather than images, the step also finds the real validation frame whose embedding is closest to each prediction. It measures how often this frame is within two frames of the true future, compared with the no-change guess, and displays example frames for different steering inputs. The sign convention of the steering value is not documented, so results refer to positive and negative steering.

In [ ]:
# Step 32: action response test and closest-frame pictures

enc_v4.eval()
pred_v4.eval()
torch.manual_seed(0)

now_idx, next_idx, val_acts = val_index_all
n_val = len(now_idx)
val_eps_arr = np.array(val_example_eps_all)

# embed every validation example once
with torch.no_grad():
    Z_now, Z_next = [], []
    for s in range(0, n_val, 512):
        ids = torch.arange(s, min(s + 512, n_val), device=device)
        f, a, nx = get_batch(val_index_all, ids)
        Z_now.append(enc_v4(f))
        Z_next.append(enc_v4(nx))
    Z_now, Z_next = torch.cat(Z_now), torch.cat(Z_next)


def pred_error(actions, ids):
    with torch.no_grad():
        return ((pred_v4(Z_now[ids], actions) - Z_next[ids]) ** 2).mean(dim=1)


# Part 1: steering test
print("STEERING TEST (chance = 50%)")
steer = val_acts[:, 0]
for name, mask, positive_is_right in [("steer + moments", steer > 0.1, True),
                                      ("steer - moments", steer < -0.1, False)]:
    ids = torch.nonzero(mask).squeeze(1)
    if len(ids) == 0:
        print(f"  {name}: no examples")
        continue
    a_pos = val_acts[ids].clone(); a_pos[:, 0] = 0.2
    a_neg = val_acts[ids].clone(); a_neg[:, 0] = -0.2
    e_pos, e_neg = pred_error(a_pos, ids), pred_error(a_neg, ids)
    correct = (e_pos < e_neg) if positive_is_right else (e_neg < e_pos)
    print(f"  {name}: {len(ids)} examples | matching steering fits better in {correct.float().mean():.0%}")

# Part 2: throttle vs brake test
print("\nTHROTTLE vs BRAKE TEST (chance = 50%)")
thr, brk = val_acts[:, 1], val_acts[:, 2]
for name, mask, brake_is_right in [("braking moments", brk > 0.5, True),
                                   ("accelerating moments", (thr > 0.5) & (brk < 0.05), False)]:
    ids = torch.nonzero(mask).squeeze(1)
    if len(ids) == 0:
        print(f"  {name}: no examples")
        continue
    a_brake = val_acts[ids].clone(); a_brake[:, 1] = 0.0; a_brake[:, 2] = 1.0
    a_gas = val_acts[ids].clone(); a_gas[:, 1] = 0.6; a_gas[:, 2] = 0.0
    e_brake, e_gas = pred_error(a_brake, ids), pred_error(a_gas, ids)
    correct = (e_brake < e_gas) if brake_is_right else (e_gas < e_brake)
    print(f"  {name}: {len(ids)} examples | matching action fits better in {correct.float().mean():.0%}")


# Part 3: closest real frame to each prediction
def closest_frame(q):
    return torch.cdist(q, Z_next).argmin(dim=1)   # position among validation future frames


def frame_label(pos):
    ep = val_eps_arr[pos]
    return ep, int(next_idx[pos].item()) - offsets[ep]


sample = torch.randperm(n_val, device=device)[:2000]
with torch.no_grad():
    match_model = closest_frame(pred_v4(Z_now[sample], val_acts[sample]))
    match_still = closest_frame(Z_now[sample])


def hit_rate(match):
    same_ep = torch.tensor(val_eps_arr[sample.cpu().numpy()] == val_eps_arr[match.cpu().numpy()],
                           device=device)
    close = (next_idx[match] - next_idx[sample]).abs() <= 2
    return (same_ep & close).float().mean().item()


print("\nCLOSEST-FRAME TEST (2,000 validation examples)")
print(f"  model prediction lands within 2 frames of the true future: {hit_rate(match_model):.0%}")
print(f"  no-change guess lands within 2 frames of the true future:  {hit_rate(match_still):.0%}")

# pictures: 3 moments with strong steering
strong = torch.argsort(steer.abs(), descending=True)[:300]
picks = strong[torch.randperm(len(strong), device=device)[:3]]


def show(ax, global_idx, title):
    ax.imshow(all_frames[global_idx].permute(1, 2, 0).cpu())
    ax.set_title(title, fontsize=8)
    ax.axis("off")


fig, axes = plt.subplots(3, 5, figsize=(20, 8))
for row, i in enumerate(picks):
    i = i.unsqueeze(0)
    ep, t_true = frame_label(i.item())
    show(axes[row, 0], now_idx[i].item(), f"current\n{ep}, steer {val_acts[i, 0].item():+.2f}")
    show(axes[row, 1], next_idx[i].item(), f"true future\nframe {t_true}")
    for col, (label, s_val) in enumerate([("recorded action", None), ("steer +0.2", 0.2), ("steer -0.2", -0.2)], 2):
        a = val_acts[i].clone()
        if s_val is not None:
            a[:, 0] = s_val
        with torch.no_grad():
            m = closest_frame(pred_v4(Z_now[i], a)).item()
        m_ep, m_t = frame_label(m)
        where = f"{m_t - t_true:+d} frames from true future" if m_ep == ep else f"other episode ({m_ep})"
        show(axes[row, col], next_idx[m].item(), f"closest to prediction: {label}\n{where}")
plt.suptitle("v4: current frame, true future, and the real frames closest to the model's predictions")
plt.tight_layout()
plt.show()

In [ ]:
# Step 32 pictures again, using the original full-resolution frames for display

import bisect

ep_starts = sorted((o, ep) for ep, o in offsets.items())
starts = [o for o, _ in ep_starts]
zip_full = zipfile.ZipFile(dataset_zip_path)


def global_to_ep_t(g):
    k = bisect.bisect_right(starts, g) - 1
    o, ep = ep_starts[k]
    return ep, g - o


def show_full(ax, global_idx, title):
    ep, t = global_to_ep_t(global_idx)
    with zip_full.open(f"data/{ep}/frames/{t:06d}.jpg") as f:
        ax.imshow(Image.open(f).convert("RGB"))
    ax.set_title(title, fontsize=8)
    ax.axis("off")


fig, axes = plt.subplots(3, 5, figsize=(22, 9))
for row, i in enumerate(picks):
    i = i.unsqueeze(0)
    ep, t_true = frame_label(i.item())
    show_full(axes[row, 0], now_idx[i].item(), f"current\n{ep}, steer {val_acts[i, 0].item():+.2f}")
    show_full(axes[row, 1], next_idx[i].item(), f"true future\nframe {t_true}")
    for col, (label, s_val) in enumerate([("recorded action", None), ("steer +0.2", 0.2), ("steer -0.2", -0.2)], 2):
        a = val_acts[i].clone()
        if s_val is not None:
            a[:, 0] = s_val
        with torch.no_grad():
            m = closest_frame(pred_v4(Z_now[i], a)).item()
        m_ep, m_t = frame_label(m)
        where = f"{m_t - t_true:+d} frames from true future" if m_ep == ep else f"other episode ({m_ep})"
        show_full(axes[row, col], next_idx[m].item(), f"closest to prediction: {label}\n{where}")
plt.suptitle("v4: current frame, true future, and real frames closest to the model's predictions (full resolution)")
plt.tight_layout()
plt.show()

### Step 33: Diagnosing the Throttle and Brake Result

This step investigates why the model does not distinguish braking from accelerating. First, it checks in the recorded data whether braking is followed by a decrease in the first state value, which is assumed to be the vehicle speed. Second, it fits a simple linear model to test whether the current speed can be read from the v4 embedding of a single frame. Third, it tests whether the speed change can be read from the difference between the current and future embeddings. Validation episodes are split in half by episode, with one half used to fit the linear models and the other used to test them.

In [ ]:
# Step 33: why can't the model tell braking from throttle?

# load the state arrays for every episode
episode_states = {}
with zipfile.ZipFile(dataset_zip_path) as z:
    for ep in all_eps:
        with z.open(f"data/{ep}/log.npz") as f:
            episode_states[ep] = np.load(BytesIO(f.read()))["states"].astype(np.float32)


def speed_arrays(episodes, gap):
    # speed now and speed change over the gap, in the same order as build_index
    now, change = [], []
    for ep in episodes:
        s = episode_states[ep][:, 0]
        n = len(episode_actions[ep]) - gap
        now.append(s[:n])
        change.append(s[gap:gap + n] - s[:n])
    return np.concatenate(now), np.concatenate(change)


# 1. does braking slow the car in the data?
_, speed_change_train = speed_arrays(train_eps_all, FRAME_GAP)
acts_train = train_index_all[2].cpu().numpy()
braking = acts_train[:, 2] > 0.5
accelerating = (acts_train[:, 1] > 0.5) & (acts_train[:, 2] < 0.05)
print("1. SPEED CHANGE IN THE DATA (over 0.25 s)")
print(f"   braking moments:      average speed change {speed_change_train[braking].mean():+.3f}")
print(f"   accelerating moments: average speed change {speed_change_train[accelerating].mean():+.3f}")

# 2 and 3. can speed and speed change be read from the embeddings?
speed_now_val, speed_change_val = speed_arrays(val_eps_all, FRAME_GAP)
speed_now_val = torch.tensor(speed_now_val, device=device)
speed_change_val = torch.tensor(speed_change_val, device=device)

# split validation episodes in half by episode
half = set(val_eps_all[: len(val_eps_all) // 2])
fit_mask = torch.tensor([ep in half for ep in val_example_eps_all], device=device)


def linear_probe(X, y):
    X = torch.cat([X, torch.ones(len(X), 1, device=device)], dim=1)
    Xf, yf = X[fit_mask], y[fit_mask]
    w = torch.linalg.solve(Xf.T @ Xf + 1e-2 * torch.eye(X.shape[1], device=device), Xf.T @ yf)
    Xt, yt = X[~fit_mask], y[~fit_mask]
    pred = Xt @ w
    r2 = 1 - ((pred - yt) ** 2).sum() / ((yt - yt.mean()) ** 2).sum()
    return r2.item()


print("\n2. CAN THE CURRENT SPEED BE READ FROM ONE FRAME'S EMBEDDING?")
print(f"   R2 = {linear_probe(Z_now, speed_now_val):.2f}   (1 = perfectly, 0 = not at all)")

print("\n3. CAN THE SPEED CHANGE BE READ FROM THE EMBEDDING DIFFERENCE?")
print(f"   R2 = {linear_probe(Z_next - Z_now, speed_change_val):.2f}")

**Result:** The recorded data is consistent: braking was followed by an average decrease of 0.85 in the first state value over 0.25 seconds, and accelerating by an average increase of 0.39, which confirms that this value represents the vehicle speed. However, the speed could not be predicted from the v4 embedding of a single frame (R² = -0.81), and the speed change could not be predicted from the difference between the current and future embeddings (R² = -0.38). Negative R² values indicate that the linear models performed worse than predicting the average. The embeddings therefore do not represent the vehicle speed or its change, so the prediction loss gives the model no reason to learn the effect of braking or accelerating. This suggests that the encoder needs several consecutive frames, so that motion and speed are visible in its input.

### Step 34: Model with Motion Input (v5)

This step addresses the results of Steps 32 and 33. The encoder now receives three consecutive frames, taken 0.5 seconds before, 0.25 seconds before, and at the current moment, so that motion and speed can be represented in the embedding. The current vehicle speed is also given to the predictor alongside the action. The prediction horizon is increased from five to ten frames (0.5 seconds), so that changes in speed have a larger visual effect, and braking moments are sampled three times as often during training. All other settings are the same as v4. Because the prediction horizon differs from v4, the gain over the no-change baseline is not directly comparable between the two versions. The steering and braking tests and the speed probes are used for comparison instead.

In [ ]:
# Step 34: v5 = 3-frame input + speed input + 10-frame horizon + more braking examples

GAP5 = 10    # predict 10 frames ahead (0.5 s)
STEP5 = 5    # spacing between the 3 input frames (0.25 s)
OFFS_NOW = torch.tensor([-2 * STEP5, -STEP5, 0], device=device)
OFFS_NEXT = OFFS_NOW + GAP5


def build_index_v5(episodes):
    last, act, speed, eps_list = [], [], [], []
    for ep in episodes:
        o, acts, s = offsets[ep], episode_actions[ep], episode_states[ep][:, 0]
        for t in range(2 * STEP5, len(acts) - GAP5):
            last.append(o + t)
            act.append(acts[t:t + GAP5].mean(axis=0))
            speed.append(s[t])
            eps_list.append(ep)
    return (torch.tensor(last, device=device),
            torch.tensor(np.array(act), device=device),
            torch.tensor(np.array(speed), device=device),
            eps_list)


def get_batch_v5(index, ids):
    last, act, speed, _ = index
    base = last[ids][:, None]
    f = all_frames[base + OFFS_NOW].flatten(1, 2).float() / 255    # (B, 9, H, W)
    n = all_frames[base + OFFS_NEXT].flatten(1, 2).float() / 255
    return f, act[ids], speed[ids], n


class EncoderV5(EncoderV2):
    def __init__(self, dim=EMBED_DIM):
        super().__init__(dim)
        self.conv[0] = nn.Conv2d(9, 32, 4, stride=2, padding=1)   # 3 frames x 3 colours


class PredictorV5(nn.Module):
    def __init__(self, dim=EMBED_DIM, dropout=0.1):
        super().__init__()
        self.action_net = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64))  # action + speed
        self.net = nn.Sequential(
            nn.Linear(dim + 64, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, z, a, speed):
        extra = torch.cat([a, speed[:, None] / 10.0], dim=1)
        return z + self.net(torch.cat([z, self.action_net(extra)], dim=1))


train_v5 = build_index_v5(train_eps_all)
val_v5 = build_index_v5(val_eps_all)
n_train_v5 = len(train_v5[0])
print("Train examples:", n_train_v5, "| Val examples:", len(val_v5[0]))

# braking moments are sampled 3 times as often
sample_weights = 1 + 2 * (train_v5[1][:, 2] > 0.5).float()

torch.manual_seed(0)
np.random.seed(0)
enc_v5 = EncoderV5().to(device)
pred_v5 = PredictorV5().to(device)
params_v5 = list(enc_v5.parameters()) + list(pred_v5.parameters())
opt_v5 = torch.optim.AdamW(params_v5, lr=1e-3, weight_decay=0.05)

BATCH = 64
STEPS = int(4 * n_train_v5 / BATCH)
WARMUP = 250
EVAL_EVERY = 1000
sched_v5 = torch.optim.lr_scheduler.LambdaLR(opt_v5, lr_factor)


def embed_v5(index):
    enc_v5.eval()
    n = len(index[0])
    zn, zx = [], []
    with torch.no_grad():
        for s in range(0, n, 512):
            ids = torch.arange(s, min(s + 512, n), device=device)
            f, a, sp, nx = get_batch_v5(index, ids)
            zn.append(enc_v5(f)); zx.append(enc_v5(nx))
    enc_v5.train()
    return torch.cat(zn), torch.cat(zx)


def evaluate_v5(index):
    z_now, z_next = embed_v5(index)
    pred_v5.eval()
    _, acts, speed, eps_list = index
    with torch.no_grad():
        def err(p):
            return ((p - z_next) ** 2).mean(dim=1).cpu().numpy()
        g = torch.Generator(device=device).manual_seed(0)
        shuffled = acts[torch.randperm(len(acts), device=device, generator=g)]
        df = pd.DataFrame({
            "episode": eps_list,
            "model": err(pred_v5(z_now, acts, speed)),
            "no_change": err(z_now),
            "shuffled": err(pred_v5(z_now, shuffled, speed)),
            "zero": err(pred_v5(z_now, torch.zeros_like(acts), speed)),
        })
    pred_v5.train()
    return df, z_now, z_next


print(f"Training v5 for {STEPS} steps")
start = time.time()
for step in range(1, STEPS + 1):
    ids = torch.multinomial(sample_weights, BATCH, replacement=True)
    f, a, sp, n = get_batch_v5(train_v5, ids)
    f, a, n = augment(f, a, n)

    z_now, z_next = enc_v5(f), enc_v5(n)
    pred_loss = F.mse_loss(pred_v5(z_now, a, sp), z_next)
    reg_loss = sigreg(torch.cat([z_now, z_next], dim=0))
    loss = pred_loss + LAMBDA * reg_loss

    opt_v5.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_v5, 1.0)
    opt_v5.step()
    sched_v5.step()

    if step % EVAL_EVERY == 0 or step == STEPS:
        df, _, _ = evaluate_v5(val_v5)
        beat = (df["model"] < df["no_change"]).mean()
        print(f"step {step:5d} | model {df['model'].mean():.4f} | no change {df['no_change'].mean():.4f} | "
              f"shuffled {df['shuffled'].mean():.4f} | beats no-change {beat:.0%} | {time.time() - start:.0f}s")

torch.save({"encoder": enc_v5.state_dict(), "predictor": pred_v5.state_dict()},
           f"{CACHE_DIR}/jepa_v5_step34.pt")
print("\nModel saved to Drive")

# overall results on all 47 validation episodes
v5_df, Z5_now, Z5_next = evaluate_v5(val_v5)
print("\nAll 47 validation episodes (0.5 s horizon):")
print(pd.DataFrame([summarize(v5_df, "v5 (Step 34)")]).to_string(index=False))

# steering and brake tests, same as Step 32
_, acts_v, speed_v, _ = val_v5


def err_v5(a, ids):
    with torch.no_grad():
        return ((pred_v5(Z5_now[ids], a, speed_v[ids]) - Z5_next[ids]) ** 2).mean(dim=1)


pred_v5.eval()
print("\nSTEERING TEST (chance = 50%)")
for name, mask, pos_right in [("steer + moments", acts_v[:, 0] > 0.1, True),
                              ("steer - moments", acts_v[:, 0] < -0.1, False)]:
    ids = torch.nonzero(mask).squeeze(1)
    a_pos = acts_v[ids].clone(); a_pos[:, 0] = 0.2
    a_neg = acts_v[ids].clone(); a_neg[:, 0] = -0.2
    e_pos, e_neg = err_v5(a_pos, ids), err_v5(a_neg, ids)
    ok = (e_pos < e_neg) if pos_right else (e_neg < e_pos)
    print(f"  {name}: {len(ids)} examples | matching steering fits better in {ok.float().mean():.0%}")

print("\nTHROTTLE vs BRAKE TEST (chance = 50%)")
for name, mask, brake_right in [("braking moments", acts_v[:, 2] > 0.5, True),
                                ("accelerating moments", (acts_v[:, 1] > 0.5) & (acts_v[:, 2] < 0.05), False)]:
    ids = torch.nonzero(mask).squeeze(1)
    a_b = acts_v[ids].clone(); a_b[:, 1] = 0.0; a_b[:, 2] = 1.0
    a_g = acts_v[ids].clone(); a_g[:, 1] = 0.6; a_g[:, 2] = 0.0
    e_b, e_g = err_v5(a_b, ids), err_v5(a_g, ids)
    ok = (e_b < e_g) if brake_right else (e_g < e_b)
    print(f"  {name}: {len(ids)} examples | matching action fits better in {ok.float().mean():.0%}")
pred_v5.train()

# speed checks, same as Step 33
speed_change_v5 = []
for ep in val_eps_all:
    s = episode_states[ep][:, 0]
    for t in range(2 * STEP5, len(episode_actions[ep]) - GAP5):
        speed_change_v5.append(s[t + GAP5] - s[t])
speed_change_v5 = torch.tensor(np.array(speed_change_v5), device=device)
fit_mask = torch.tensor([ep in half for ep in val_v5[3]], device=device)

print("\nSPEED CHECKS (R2: 1 = perfectly, 0 or below = not at all)")
print(f"  current speed from embedding:          R2 = {linear_probe(Z5_now, speed_v):.2f}   (v4: -0.81)")
print(f"  speed change from embedding difference: R2 = {linear_probe(Z5_next - Z5_now, speed_change_v5):.2f}   (v4: -0.38)")

**Result:**

 The v5 model trained steadily on the full dataset, and the share of validation examples where it beat the no-change baseline rose from 48 to 58 percent. At the 0.5 second horizon, its error was 17.6 percent lower than the baseline (95 percent interval 14.4 to 23.8 percent), and shuffling the actions increased the error by 32.1 percent. Because v5 predicts twice as far ahead as v4, these values are not directly comparable. The steering test remained strong at 90 percent for both directions. The throttle and brake test improved from 48 to 49 percent in v4 to 56 to 57 percent, which is above chance but only modestly. The speed probes remained negative (R² of -0.63 for the current speed and -0.64 for the speed change), so the embeddings still do not linearly represent the vehicle speed, even with three input frames. The improvement in the braking test is therefore likely due to the speed value given directly to the predictor, similar to the use of the robot state in V-JEPA 2-AC. Further improvement is expected from higher-resolution frames, a larger encoder, and longer training.

### Step 35: Results Visualization

This step summarizes the main results in three charts. The first shows each version's improvement over the no-change baseline with its 95 percent interval, separated by validation set because v1 to v3 were evaluated on 11 episodes and v4 and v5 on all 47. The second compares the steering and throttle-brake tests for v4 and v5 against the 50 percent chance level. The third shows the v4 improvement for each weather condition. All values are taken from the earlier steps.

In [ ]:
# Step 35: results charts (values from Steps 28 to 34)

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#333333", "#777777", "#e6e6e6"


def tidy(ax):
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


# 1. gain vs no-change with 95% intervals
sets = [
    ("11 validation episodes (0.25 s ahead)",
     ["v1", "v2", "v3", "v4"], [17.1, -18.4, 16.8, 18.2],
     [8.4, -41.4, 10.7, 11.8], [23.6, 8.1, 23.2, 18.9]),
    ("All 47 validation episodes",
     ["v3", "v4", "v5*"], [18.5, 20.7, 17.6],
     [14.8, 16.4, 14.4], [24.2, 25.9, 23.8]),
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (title, names, gain, low, high) in zip(axes, sets):
    x = np.arange(len(names))
    err = [np.array(gain) - np.array(low), np.array(high) - np.array(gain)]
    ax.errorbar(x, gain, yerr=err, fmt="o", color=BLUE, ecolor=BLUE,
                elinewidth=2, capsize=5, markersize=8)
    for xi, g in zip(x, gain):
        ax.annotate(f"{g:.1f}%", (xi, g), textcoords="offset points", xytext=(10, -4), color=INK, fontsize=9)
    ax.axhline(0, color=MUTED, linewidth=1)
    ax.set_xticks(x, names)
    ax.set_xlim(-0.5, len(names) - 0.5)
    ax.set_title(title, color=INK, fontsize=11)
    tidy(ax)
axes[0].set_ylabel("Error reduction vs no-change (%)", color=INK)
fig.suptitle("Improvement over the no-change baseline, with 95% intervals", color=INK)
fig.text(0.5, -0.02, "* v5 predicts 0.5 s ahead, so its value is not directly comparable with v1 to v4.",
         ha="center", color=MUTED, fontsize=9)
plt.tight_layout()
plt.show()

# 2. action tests, v4 vs v5
tests = ["Steering", "Throttle vs brake"]
v4_scores = [93.5, 48.5]   # averages of the two directions in Step 32
v5_scores = [90.0, 56.5]   # averages of the two directions in Step 34
x = np.arange(len(tests))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w / 2 - 0.01, v4_scores, w, color=BLUE, label="v4")
b2 = ax.bar(x + w / 2 + 0.01, v5_scores, w, color=ORANGE, label="v5")
for bars in (b1, b2):
    for bar in bars:
        ax.annotate(f"{bar.get_height():.0f}%", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    textcoords="offset points", xytext=(0, 3), ha="center", color=INK, fontsize=9)
ax.axhline(50, color=MUTED, linestyle="--", linewidth=1)
ax.text(0.5, 52, "chance (50%)", color=MUTED, fontsize=9, ha="center")
ax.set_xticks(x, tests)
ax.set_ylim(0, 105)
ax.set_ylabel("Matching action fits better (%)", color=INK)
ax.set_title("Does the model respond correctly to each action?", color=INK, fontsize=11)
ax.legend(frameon=False)
tidy(ax)
plt.tight_layout()
plt.show()

# 3. v4 gain by weather (all 47 validation episodes)
weather_gain = (by_weather_v4["gain_vs_no_change"] * 100).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(weather_gain.index, weather_gain.values, color=BLUE, height=0.6)
for i, (name, g) in enumerate(weather_gain.items()):
    eps = by_weather_v4.loc[name, "episodes"]
    ax.text(g + 0.5, i, f"{g:.0f}%  ({eps} ep)", va="center", color=INK, fontsize=9)
ax.set_xlabel("Error reduction vs no-change (%)", color=INK)
ax.set_title("v4 improvement by weather (all 47 validation episodes)", color=INK, fontsize=11)
ax.set_xlim(0, weather_gain.max() + 8)
tidy(ax)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color=GRID, linewidth=0.8)
plt.tight_layout()
plt.show()

**Result:**

All versions except v2 improved on the no-change baseline by about 17 to 21 percent, while v2 fell below it because of overfitting. Both v4 and v5 respond strongly to steering (90 to 94 percent), but only v5 is above chance on throttle and brake, and only slightly (56 percent). v4 improved on the baseline in every weather condition, from 11 to 31 percent.

### Step 36: Summary

**Data.** The dataset contains 297 CARLA driving episodes with a single front-facing camera. This was confirmed for every episode: each has one image folder, one image per timestamp, and the same seven log arrays, none of them camera-related. Each frame is paired with three actions (steering, throttle, and brake), and the dataset's own split field divides the episodes into 250 for training and 47 for validation. There is no separate test set, so all results below are validation results.

**Approach.** The model is a simplified action-conditioned JEPA based on LeWorldModel. An encoder converts frames into embeddings, and a predictor uses the current embedding and the action to predict the embedding of a future frame. Training combines a prediction loss with SIGReg, which prevents collapse. Each version was compared with a no-change baseline, which assumes the scene stays the same, and with shuffled actions, to check whether the model uses the action information.

**Model versions**

| Version | Data | Main change | Gain vs no-change | Shuffled-action increase | Key finding |
|---|---|---|---|---|---|
| v0 (Step 24) | 20% subset | First attempt | Did not beat baseline | 0% | Ignored actions |
| v1 (Step 26) | 20% subset | Residual predictor, spatial layout | 17.1% | 40.8% | First working model |
| v2 (Step 29) | 20% subset | Longer training | -18.4% | 28.9% | Overfitting |
| v3 (Step 30) | 20% subset | Augmentation, dropout | 16.8% | 30.6% | Overfitting removed |
| v4 (Step 31) | Full dataset | All 250 training episodes | 20.7% | 41.6% | Best short-range accuracy |
| v5 (Step 34) | Full dataset | Three input frames, speed input, 0.5 s horizon | 17.6% | 32.1% | Responds to all three actions |

The values for v1 to v3 are measured on the 11 validation episodes of the smoke-test subset. The values for v4 and v5 are measured on all 47 validation episodes. v1 to v4 predict 0.25 seconds ahead and v5 predicts 0.5 seconds ahead, so their gains are not directly comparable.

**Main findings**
- The pipeline runs end to end, and the smoke test on the 20 percent subset passed.
- The final models reliably outperform the no-change baseline and depend on the action information.
- More training data improved the model, while longer training on the small subset caused overfitting.
- The model clearly understands steering, choosing the correct steering direction in 90 to 94 percent of cases.
- Understanding of braking and accelerating is weak: 48 to 49 percent for v4 (chance level) and 56 to 57 percent for v5.
- Speed could not be read from the embeddings, which suggests that the encoder does not yet capture motion and speed.
- With the full validation set, the model outperformed the baseline in all 11 weather conditions present.

**Limitations**
- Only a front camera is available in this dataset.
- Frames are reduced to 72 by 128 pixels, which removes fine detail.
- The model is small, and each version was trained once with a single random seed.
- There is no separate test set.
- Planning with model predictive control has not been tested, because no simulator is available in Colab.

**Next steps**
- Test the changes made in v5 one at a time to see which ones matter most.
- Set aside a held-out test set for final evaluation.
- Use the model for offline action selection with the cross-entropy method, as in LeWorldModel and V-JEPA 2-AC.
- Move to HPRC for higher-resolution frames, a larger encoder, a transformer predictor, and longer training.
- Run front and back camera experiments with nuScenes.